<a href="https://colab.research.google.com/github/whiteraay/research/blob/main/colab/01_operator_workspace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APD benchmark — Operator workspace

You run the code; the chat model (Claude / ChatGPT / Gemini, free web chat) writes it. Data stays on Drive, mounted read-only. Your outputs are saved to `MyDrive/apd_outputs_<model>/` and survive Colab resets.

**Every session:** run cell 1. **New chat:** cell 2 (first time) or cell 4 (resume). **Then loop:** paste the model's cells into the empty cells at the bottom, run, copy the part between `COPY FROM HERE` and `END OF COPY` back into the chat.

**Before the first session**
- Drive: open the shared **`apd_benchmark`** folder → ⋮ → Organize → **Add shortcut** → My Drive.
- Colab: Tools → Settings → turn **off** AI-powered code completions / generation. Never use Colab's built-in Gemini panel. The code must come only from your chat.
- Your chat: turn off "use my conversations to improve the model" (or equivalent).

In [3]:
#@title 1. Setup — run at the start of EVERY Colab session
# Mount Google Drive (NOT upload) and get the project code.
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, shutil, subprocess
REPO = "/content/research"
if not os.path.exists(REPO):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # optional Colab secret (key icon on the left)
    except Exception:
        pass
    if token:
        subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/whiteraay/research.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/whiteraay/research.git"])
        print("code: cloned from GitHub")
    else:
        cands = ["/content/drive/MyDrive/apd_benchmark/code/research",
                 *glob.glob("/content/drive/.shortcut-targets-by-id/*/apd_benchmark/code/research")]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            raise SystemExit("No GITHUB_TOKEN secret and no code copy at MyDrive/apd_benchmark/code/research. "
                             "Add the apd_benchmark shortcut to My Drive (operators) or add a GITHUB_TOKEN secret.")
        shutil.copytree(src, REPO)
        print("code: copied from", src)
sys.path.insert(0, REPO + "/src")
from apd_bench import colab as apd

MODEL = "claude"     # <- operator 1: "claude" | operator 2: "gpt" | operator 3: "gemini"
apd.setup_workspace(MODEL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Note: output folders of ['gpt'] are visible from this account (shared lab account?). That is allowed, but never open them or mention them in the chat until scoring is done.
Workspace ready for claude.
  model         claude
  repo          /content/research
  data          /content/drive/MyDrive/apd_benchmark/data
  outputs       /content/drive/MyDrive/apd_outputs_claude
  train_videos  146
  test_videos   62
  audit         True
  state         False


{'model': 'claude',
 'repo': '/content/research',
 'data': '/content/drive/MyDrive/apd_benchmark/data',
 'outputs': '/content/drive/MyDrive/apd_outputs_claude',
 'train_videos': 146,
 'test_videos': 62,
 'audit': True,
 'state': False}

## 2. First chat message: charter + execution protocol
Copy everything between the markers into a **new** chat. Wait for the model's acknowledgement.

In [6]:
#@title 2. Print charter + protocol
apd.prompt(0)

==================== COPY FROM HERE INTO THE CHAT ====================
# PROJECT CHARTER — read fully before doing anything

## Your role
You are the lead data scientist on a clinical machine-learning study. You will design, implement, evaluate and document a complete method for identifying participants at elevated acute psychological distress risk from video-recorded psychological interviews. You decide the methodology: which features, which pre-trained models, which classifier, how to fuse channels. Nobody will tell you what to use; your choices and the reasons for them are part of what is being studied.

Two other AI systems are receiving this same charter, the same data, the same compute and the same human operator protocol. Their work and yours will be compared on identical held-out sessions. Make the choices you believe are scientifically best, not the choices you think a comparison wants to see.

## How you work in this study (important)
You cannot run code and you cannot see th

## 3. Phase prompts
Send the next phase only after the model says **PHASE N COMPLETE** and `apd.ls()` shows its files.

In [7]:
#@title 3. Print a phase prompt
PHASE = 1   # 1 audit · 2 proposal · 3 build · 4 evaluate · 5 affect · 6 report
apd.prompt(PHASE)

==================== COPY FROM HERE INTO THE CHAT ====================
# PHASE 1 — DATA AUDIT

Before proposing any method, understand the data — through code, since you cannot look at it. Deliverable: `outputs/claude/audit_notes.md` (plus the small scripts you used, under `src/audit/`). Do not extract model features or train anything in this phase.

1. **Inventory.** Write a script that prints: sessions per cohort (train) and in total (test); participants per cohort; sessions per participant; file formats, resolution, frame rate; duration distribution (min, quartiles, max) for train and test.
2. **Recording conditions.** Read `data/audit.csv`. For duration_s, mean_luminance, face_detect_rate and audio_snr_db print the distribution, and in the training set test whether each differs between cohorts (effect size + a simple test). Also fit a cross-validated (grouped by participant) logistic regression on audit.csv columns alone and print its AUROC: this is the **confound floor** every lat

## 4. Usage limit hit / chat broke / new day
Open a **new** chat with the **same** model and paste this one message. It contains the charter, the protocol, STATE.md, decisions.md and the file list.

In [4]:
#@title 4. Print resume packet
apd.resume_packet()

==================== COPY FROM HERE INTO THE CHAT ====================
# PROJECT CHARTER — read fully before doing anything

## Your role
You are the lead data scientist on a clinical machine-learning study. You will design, implement, evaluate and document a complete method for identifying participants at elevated acute psychological distress risk from video-recorded psychological interviews. You decide the methodology: which features, which pre-trained models, which classifier, how to fuse channels. Nobody will tell you what to use; your choices and the reasons for them are part of what is being studied.

Two other AI systems are receiving this same charter, the same data, the same compute and the same human operator protocol. Their work and yours will be compared on identical held-out sessions. Make the choices you believe are scientifically best, not the choices you think a comparison wants to see.

## How you work in this study (important)
You cannot run code and you cannot see th

'# PROJECT CHARTER — read fully before doing anything\n\n## Your role\nYou are the lead data scientist on a clinical machine-learning study. You will design, implement, evaluate and document a complete method for identifying participants at elevated acute psychological distress risk from video-recorded psychological interviews. You decide the methodology: which features, which pre-trained models, which classifier, how to fuse channels. Nobody will tell you what to use; your choices and the reasons for them are part of what is being studied.\n\nTwo other AI systems are receiving this same charter, the same data, the same compute and the same human operator protocol. Their work and yours will be compared on identical held-out sessions. Make the choices you believe are scientifically best, not the choices you think a comparison wants to see.\n\n## How you work in this study (important)\nYou cannot run code and you cannot see the data. A human operator runs your code for you in Google Cola

## 5. Log what happens in the chat
Record the exact model name the chat shows, and every limit, downgrade, refusal or new chat. These are part of the results.

In [9]:
#@title 5. Log a chat event
EVENT = "start"        # start | limit | resume | downgrade | refusal | error | phase_done | gpu_on | gpu_unavailable
MODEL_SHOWN = "Sonnet 5.5"       # exactly as the chat UI shows it, e.g. the model picker label
DETAIL = ""
apd.log_chat(EVENT, MODEL_SHOWN, DETAIL)

logged: chat start Sonnet 5.5


## 6. Paste the model's cells below
One model cell per Colab cell, in order, exactly as written. Add more cells with **+ Code**. Then paste the printed summary back into the chat.

In [5]:
apd.run("pip install -q -r outputs/claude/requirements.txt; echo '--- operator_notes'; cat outputs/claude/operator_notes.md; echo '--- requirements'; cat outputs/claude/requirements.txt; echo '--- cache'; ls -la outputs/claude/cache; echo '--- logs'; ls -t outputs/claude/logs | head -10; echo '--- last log tail'; tail -n 30 outputs/claude/logs/$(ls -t outputs/claude/logs | head -1); echo '--- audit script heads'; for f in common inventory luminance; do echo \"## $f\"; head -n 25 outputs/claude/src/audit/$f.py; done; echo '--- data/audit.csv'; head -c 600 data/audit.csv; wc -l data/audit.csv data/train/train_labels.csv")

--- operator_notes
# Operator notes — claude

| time | event | detail |
| --- | --- | --- |
| 2026-11-17 13:52:07 | colab session start | isolation OK |
| 2026-11-17 13:54:27 | prompt shown | 0 |
| 2026-11-17 13:56:17 | prompt shown | 0 |
| 2026-11-17 13:56:33 | prompt shown | 1 |
| 2026-11-17 14:15:31 | chat start | Sonnet 5.5 |
| 2026-11-18 10:58:08 | colab session start | isolation OK; other outputs visible: gpt |
| 2026-11-18 10:58:14 | resume packet built | 12,729 chars |
| 2026-11-19 07:19:07 | colab session start | isolation OK; other outputs visible: gpt |
| 2026-11-19 07:19:14 | resume packet built | 12,730 chars |
--- requirements
numpy
pandas
scipy
scikit-learn
librosa
soundfile
faster-whisper
--- cache
total 12
drwx------ 3 root root 4096 Nov 17 13:52 .
drwx------ 6 root root 4096 Nov 19 07:19 ..
drwx------ 2 root root 4096 Nov 17 14:17 audit
--- logs
cmd_0009.log
resume_20261119_071914.md
commands.csv
cmd_0008.log
cmd_0007.log
cmd_0006.log
cmd_0005.log
resume_20261118_1058

0

In [6]:
apd.run("echo '--- commands.csv'; cut -c1-200 outputs/claude/logs/commands.csv | tail -n 12; echo '--- cache/audit'; ls -la outputs/claude/cache/audit; for f in outputs/claude/cache/audit/*.csv; do echo \"## $f\"; wc -l $f; head -n 3 $f | cut -c1-250; done; echo '--- cv2'; python -c \"import cv2,sys;print(cv2.__version__, cv2.__file__, hasattr(cv2,'CascadeClassifier'))\"; pip list 2>/dev/null | grep -i -E 'opencv|numpy|faster|librosa'; echo '--- audit.csv errors'; python -c \"import pandas as pd;d=pd.read_csv('data/audit.csv');print(d.shape);print(d.error.fillna('none').str.slice(0,40).value_counts().head());print(d[['duration_s','audio_snr_db','speech_fraction']].describe().round(1).to_string());print(d.duration_tertile.value_counts().to_dict(), d.snr_tertile.value_counts().to_dict())\"; echo '--- inventory.py size/tail'; wc -l outputs/claude/src/audit/*.py; tail -n 25 outputs/claude/src/audit/inventory.py | cut -c1-180")

--- commands.csv
2026-11-17 14:15:52,"pip install -q -r outputs/claude/requirements.txt && python -c ""import numpy,pandas,scipy,sklearn,librosa,soundfile,faster_whisper,cv2;print('numpy',numpy.__version__,'pandas',pa
2026-11-17 14:17:34,python outputs/claude/src/audit/inventory.py,1,8.7,cmd_0002.log
2026-11-17 14:21:27,python outputs/claude/src/audit/inventory.py,0,35.0,cmd_0003.log
2026-11-17 14:49:56,python outputs/claude/src/audit/luminance.py,0,1685.0,cmd_0004.log
2026-11-18 10:58:43,"pip install -q -r outputs/claude/requirements.txt && python -c ""import numpy,pandas,scipy,sklearn,librosa,soundfile,faster_whisper,cv2;print('numpy',numpy.__version__,'pandas',pa
2026-11-18 10:58:51,python outputs/claude/src/audit/inventory.py,1,6.9,cmd_0006.log
2026-11-18 10:59:39,python outputs/claude/src/audit/inventory.py,0,47.8,cmd_0007.log
2026-11-18 10:59:40,python outputs/claude/src/audit/luminance.py,0,1.1,cmd_0008.log
2026-11-19 07:22:15,"pip install -q -r outputs/claude/requirements.txt; 

0

In [7]:
apd.run("cut -c1-220 outputs/claude/logs/cmd_0007.log | head -n 170; echo '--- confound_floor.csv'; cat outputs/claude/cache/audit/confound_floor.csv; echo '--- luminance NaNs'; python -c \"import pandas as pd;d=pd.read_csv('outputs/claude/cache/audit/luminance.csv');print(d.lum_mean.isna().sum(), d.lum_frames.value_counts().to_dict());print(d.lum_mean.describe().round(1).to_string())\"; echo '--- inventory luminance tertile handling'; grep -n -i 'lumin' outputs/claude/src/audit/inventory.py | cut -c1-200")

$ python outputs/claude/src/audit/inventory.py
=== 1. INVENTORY ===
audit.csv columns: ['session_id', 'path', 'duration_s', 'width', 'height', 'fps', 'audio_snr_db', 'speech_fraction', 'audio_rms_db', 'error', 'duration_tertile', 'snr_tertile', 'sid']
files by split/ext:
 split  ext 
test   .mp4     62
train  .mov      1
       .mp4    145
label rows: 146 | unique sid in labels: 146 | duplicate stems among all files: 0
labels without file: 0 | train files without label: 0 | audit rows without file: 0 | files without audit row: 0
folder-vs-csv cohort disagreements: 0

Train sessions per cohort:
 cohort
Experimental    80
Control         66
Test sessions (files): 62
Participants per cohort:
 cohort
Control         66
Experimental    80
Participants appearing in both cohorts: 0 of 146
Sessions per participant (count of participants by n sessions):
 1    146
Sessions per participant by cohort (mean, max):
               mean  max
cohort                 
Control        1.0    1
Experimental

0

In [8]:
%%writefile outputs/claude/src/audit/finish_audit.py
"""Phase 1 close-out: luminance strata (stand-in for the failed shared audit stage), luminance-vs-cohort,
confound floor incl. luminance, and generation of audit_notes.md / decisions.md / STATE.md. Label use is
restricted to descriptive audit of TRAIN sessions; test is only binned."""
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
A = "outputs/claude/cache/audit"
OUT = "outputs/claude"
SEED = 0

m = pd.read_csv(f"{A}/merged.csv")
lum = pd.read_csv(f"{A}/luminance.csv")
df = m.merge(lum, on="sid", how="left")
assert df.lum_mean.notna().all(), "missing luminance"
df["luminance_tertile"] = pd.qcut(df.lum_mean, 3, labels=["low", "mid", "high"]).astype(str)
df["log_duration"] = np.log(df.duration_s)
tr = df[df.split == "train"].copy()
te = df[df.split == "test"].copy()
tr["y"] = (tr.cohort == "Experimental").astype(int)
print(f"train {len(tr)} test {len(te)}")

# strata table for all later phases
df[["sid", "split", "duration_tertile", "luminance_tertile", "snr_tertile", "lum_mean", "lum_std"]].to_csv(f"{A}/strata.csv", index=False)

print("\n=== luminance vs cohort (train) ===")
rows = []
for c in ["lum_mean", "lum_std"]:
    a = tr.loc[tr.y == 1, c]; b = tr.loc[tr.y == 0, c]
    sd = np.sqrt((a.var() + b.var()) / 2)
    d = (a.mean() - b.mean()) / sd
    p = stats.mannwhitneyu(a, b).pvalue
    au = roc_auc_score(tr.y, tr[c])
    rows.append((c, round(d, 3), round(au, 3), round(p, 4)))
    print(f"{c}: cohen_d {d:.3f} auroc_uni {au:.3f} MWU p {p:.4f}")
t1 = pd.crosstab(tr.luminance_tertile, tr.cohort)
t2 = te.luminance_tertile.value_counts().rename("test")
tab = t1.join(t2, how="outer").fillna(0).astype(int)
p_chi = stats.chi2_contingency(t1.values)[1]
print(f"\nluminance_tertile cohort x tertile chi2 p={p_chi:.3g}")
print(tab.to_string())
print(df.groupby("luminance_tertile").lum_mean.agg(["min", "max"]).round(1).to_string())

print("\n=== confound floor with luminance (participant-grouped, 5 folds x 5 seeds, logreg) ===")
sets = {
    "audit_plus_lum": ["log_duration", "audio_snr_db", "speech_fraction", "audio_rms_db", "width", "height", "fps", "lum_mean", "lum_std"],
    "lum_only": ["lum_mean", "lum_std"],
    "duration_only": ["log_duration"],
    "duration_snr_lum": ["log_duration", "audio_snr_db", "lum_mean", "lum_std"],
}
res = []
for name, cols in sets.items():
    aucs = []
    for s in range(5):
        skf = StratifiedGroupKFold(5, shuffle=True, random_state=SEED + s)
        oof = np.zeros(len(tr))
        for a, b in skf.split(tr, tr.y, tr.participant_id):
            mdl = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
            mdl.fit(tr.iloc[a][cols], tr.y.iloc[a])
            oof[b] = mdl.predict_proba(tr.iloc[b][cols])[:, 1]
        aucs.append(roc_auc_score(tr.y, oof))
    res.append((name, len(cols), np.mean(aucs), np.std(aucs)))
    print(f"{name:18s} n_feat {len(cols)}  AUROC {np.mean(aucs):.3f} +- {np.std(aucs):.3f}")
pd.DataFrame(res, columns=["features", "n_features", "auroc_mean", "auroc_sd"]).to_csv(f"{A}/confound_floor_lum.csv", index=False)

# within-duration-tertile cohort balance (what the strata AUROCs will be up against)
print("\n=== cohort balance within duration tertile (train) ===")
print(pd.crosstab(tr.duration_tertile, tr.cohort, normalize="index").round(2).to_string())

# extreme sessions
print("\nTrain sessions with duration < 200 s or > 900 s, or audio_rms_db < -60 (sid only):")
ex = tr[(tr.duration_s < 200) | (tr.duration_s > 900) | (tr.audio_rms_db < -60)]
print(ex[["sid", "cohort", "duration_s", "audio_rms_db"]].round(1).to_string(index=False))

# ---------------- documents ----------------
cf = pd.read_csv(f"{A}/confound_floor.csv")
g = lambda f, mo: cf[(cf.features == f) & (cf.model == mo)].auroc_mean.iloc[0]
floor_all = max(r[2] for r in res if r[0] == "audit_plus_lum")
n_exp = int(tr.y.sum()); n_ctl = int((1 - tr.y).sum())
dl = {r[0]: r for r in rows}
notes = f"""# Audit notes (Phase 1)
All numbers produced by src/audit/inventory.py and src/audit/finish_audit.py (tables in cache/audit/).

## Inventory
- Train: {len(tr)} sessions ({n_exp} elevated-risk, {n_ctl} control), one session per participant, no participant in both cohorts, so participant-grouped CV equals stratified CV here. Test: {len(te)} sessions, unlabelled.
- Train labels file says "Elevated", folder says "Experimental"; they agree for all {len(tr)} sessions (0 disagreements).
- Formats: 145/146 train are 1280x720 mp4 at 30 fps; one train session is a 1920x1080 60 fps .mov. All test sessions are 1280x720 at 30 fps. No missing or duplicate files.
- Train and test recording conditions are similar (Mann-Whitney p > 0.1 for duration, SNR, speech fraction, RMS).

## Known tooling problem
data/audit.csv has no mean_luminance, luminance_std, face_detect_rate or luminance_tertile. Its `error` column for all 208 rows reports cv2 has no CascadeClassifier: the Colab image has opencv-python 5.0 and opencv-contrib 4.14 installed together and the import resolves to 5.0, which lacks it. duration_tertile and snr_tertile are present and used unchanged.
Stand-in: luminance.py computed grayscale mean/std on 12 evenly spaced frames per video (207 videos x 12 frames, 1 x 10). luminance_tertile is an equal-count tercile of lum_mean over all 208 videos (cutpoints in cache/audit/strata.csv). It is NOT the shared script's tertile, so the luminance stratum is only approximately comparable with the other systems.

## Recording conditions are confounded with cohort
- Duration: elevated-risk sessions are longer (Cohen d 0.52, univariate AUROC 0.66, participant-level p 0.0008, Holm p 0.003). Duration tertile x cohort chi2 p = 0.0002: the high-duration tertile is {int(pd.crosstab(tr.duration_tertile, tr.cohort).loc['high','Experimental'])} elevated vs {int(pd.crosstab(tr.duration_tertile, tr.cohort).loc['high','Control'])} control.
- SNR: weak difference (d 0.31, AUROC 0.59, p 0.067, Holm 0.20).
- Luminance: lum_mean d {dl['lum_mean'][1]}, AUROC {dl['lum_mean'][2]}, p {dl['lum_mean'][3]}; lum_std d {dl['lum_std'][1]}, AUROC {dl['lum_std'][2]}, p {dl['lum_std'][3]}; luminance tertile x cohort chi2 p = {p_chi:.3g}.
- Audit-column-only classifier (participant-grouped 5-fold x 5 seeds): duration alone logreg {g('duration_only','logreg'):.3f}; all 7 audit columns logreg {g('all_audit','logreg'):.3f}, shallow GBM {g('all_audit','gbm_depth2'):.3f}; permutation test p = 0.016. With luminance added: {floor_all:.3f}.
- Consequence: any model that sees interview length or features correlated with it can reach roughly 0.64-0.65 AUROC with no psychological signal. Reported AUROC must be compared with this floor, and duration-matched or duration-residualised analyses are required.
- Sample is small (146 train sessions), so CV AUROC SD of a single model is large; differences under about 0.05 are not distinguishable.

## Other flags
- One elevated-risk session is 76 s and one is 1480 s; audio_rms_db min is -85 dB (near-silent audio). These are listed by sid in the finish_audit log and will be handled explicitly (not dropped silently).
"""
open(f"{OUT}/audit_notes.md", "w").write(notes)

dec = f"""# Decisions log
## 2026-11-19 (resume, conversation 3)
- STATE.md and decisions.md did not exist although Phase 1 scripts and caches did (inventory.py run OK in cmd_0007, luminance cache complete). Trusted the file listing: Phase 1 was substantially done; this session closed it out.
- Cohort naming: labels csv "Elevated" and folder "Experimental" are the same cohort; internal label y=1 for elevated-risk.
- Shared audit.csv lacks luminance/face columns due to a cv2 5.0 vs 4.14 conflict (no CascadeClassifier). Decision: use own luminance.py stats and an own equal-count luminance_tertile (stand-in, flagged as not the shared tertile). Duration and SNR tertiles taken from audit.csv unchanged. Face detection will not use cv2 Haar cascades; the choice of detector is deferred to the proposal (Phase 2).
- Decision: because duration is confounded with cohort (AUROC 0.64-0.66 from audit columns alone), the proposal must (a) report the confound floor next to every score, (b) evaluate duration-residualised and duration-stratified performance, (c) limit/segment the analysed time window uniformly per session so extracted features do not scale with length.
- Participants are unique per session, so StratifiedGroupKFold with participant_id groups is used for all CV (fixed seed 0).
- Note: operator notes record the model for this chat as Sonnet 5.5.
"""
open(f"{OUT}/decisions.md", "w").write(dec)

state = """phase: 1 (complete after this script); entering phase 2 (proposal)
done:
- inventory/conditions/confound floor: outputs/claude/src/audit/inventory.py -> cache/audit/*.csv
- luminance stand-in: src/audit/luminance.py -> cache/audit/luminance.csv; strata: cache/audit/strata.csv
- close-out: src/audit/finish_audit.py -> audit_notes.md, decisions.md, STATE.md, cache/audit/confound_floor_lum.csv
in progress: nothing
next command: phase 2 - probe environment (GPU, cv2 state, faster-whisper model download, mediapipe/other face tools), then write proposal.md
open issues: cv2 5.0 lacks CascadeClassifier (pip has 5.0 + contrib 4.14 conflicting); requirements.txt lacks opencv pin; duration confound with cohort (floor ~0.65 AUROC); luminance tertile is a stand-in
"""
open(f"{OUT}/STATE.md", "w").write(state)
print("\nWROTE audit_notes.md decisions.md STATE.md")

Writing outputs/claude/src/audit/finish_audit.py


In [9]:
apd.run("python outputs/claude/src/audit/finish_audit.py")

train 146 test 62

=== luminance vs cohort (train) ===
lum_mean: cohen_d 0.054 auroc_uni 0.532 MWU p 0.5026
lum_std: cohen_d -0.137 auroc_uni 0.477 MWU p 0.6413

luminance_tertile cohort x tertile chi2 p=0.6
                   Control  Experimental  test
luminance_tertile                             
high                    21            28    20
low                     25            24    21
mid                     20            28    21
                     min    max
luminance_tertile              
high               131.9  188.0
low                 47.6  109.7
mid                109.9  131.9

=== confound floor with luminance (participant-grouped, 5 folds x 5 seeds, logreg) ===
audit_plus_lum     n_feat 9  AUROC 0.598 +- 0.017
lum_only           n_feat 2  AUROC 0.421 +- 0.030
duration_only      n_feat 1  AUROC 0.641 +- 0.011
duration_snr_lum   n_feat 4  AUROC 0.605 +- 0.017

=== cohort balance within duration tertile (train) ===
cohort            Control  Experimental
duration_tert

0

In [10]:
%%writefile outputs/claude/STATE.md
phase: 1 COMPLETE -> entering phase 2 (proposal)
done:
- Phase 1 audit: src/audit/{common,inventory,luminance,finish_audit}.py; tables in cache/audit/ (merged.csv, strata.csv, luminance.csv, cond_vs_cohort.csv, confound_floor.csv, confound_floor_lum.csv); audit_notes.md; decisions.md
- Data: 146 train (80 elevated-risk, 66 control, one session per participant), 62 test. 1280x720 30fps except one train 1080p60 mov.
- Confound floor: duration alone ~0.64 AUROC (participant-grouped CV), audit columns ~0.65; luminance and SNR not meaningfully confounded.
- Strata: duration and snr tertiles from data/audit.csv; luminance tertile is own equal-count stand-in (cache/audit/strata.csv) because shared audit stage failed (cv2 5.0 has no CascadeClassifier).
in progress: nothing
next command: phase 2 environment probe (GPU, cv2 state, faster-whisper model download, face/landmark and audio-embedding tools available offline-or-pip), then write proposal.md
open issues: opencv-python 5.0 and opencv-contrib 4.14 conflict (pin one in requirements.txt); duration confound with cohort (proposal must use uniform time windows and report duration-residualised/stratified results); 6 extreme train sessions (all elevated-risk): sids s1d5f1357 s2e121fb7 s46a66918 sa6fe9361 sd49cc219 sf0c9b7ae

Overwriting outputs/claude/STATE.md


In [11]:
apd.run("printf '%s\\n' '- Note: audit_plus_lum logreg AUROC 0.598 is below duration-only 0.641 because width/height/fps add noise (one 1080p60 session; format_only is below chance). Quoted floor for comparison: about 0.64-0.65 from duration.' '- All 6 extreme train sessions (76 s, 181 s, 765 s with near-silent audio, 910 s, 1348 s, 1480 s) are elevated-risk. Not dropped; handled with uniform time windows and flagged in failures.csv or strata analyses.' >> outputs/claude/decisions.md; tail -n 4 outputs/claude/decisions.md")

- Participants are unique per session, so StratifiedGroupKFold with participant_id groups is used for all CV (fixed seed 0).
- Note: operator notes record the model for this chat as Sonnet 5.5.
- Note: audit_plus_lum logreg AUROC 0.598 is below duration-only 0.641 because width/height/fps add noise (one 1080p60 session; format_only is below chance). Quoted floor for comparison: about 0.64-0.65 from duration.
- All 6 extreme train sessions (76 s, 181 s, 765 s with near-silent audio, 910 s, 1348 s, 1480 s) are elevated-risk. Not dropped; handled with uniform time windows and flagged in failures.csv or strata analyses.

==================== COPY FROM HERE INTO THE CHAT ====================
$ printf '%s\n' '- Note: audit_plus_lum logreg AUROC 0.598 is below duration-only 0.641 because width/height/fps add noise (one 1080p60 session; format_only is below chance). Quoted floor for comparison: about 0.64-0.65 from duration.' '- All 6 extreme train sessions (76 s, 181 s, 765 s with near-silent

0

In [12]:
apd.ls()

==================== COPY FROM HERE INTO THE CHAT ====================
STATE.md  1,284 B
audit_notes.md  2,799 B
decisions.md  1,783 B
operator_notes.md  615 B
requirements.txt  65 B
cache/  (6 files, not listed)
logs/  (16 files, not listed)
src/audit/common.py  1,877 B
src/audit/finish_audit.py  9,710 B
src/audit/inventory.py  9,161 B
src/audit/luminance.py  1,744 B
src/audit/__pycache__/common.cpython-313.pyc  4,161 B
==================== END OF COPY ====================


## second phase

In [13]:
%%writefile outputs/claude/src/env_probe.py
"""Phase 2 environment probe. No data access. Prints versions, hardware, installability, network reachability."""
import sys, os, shutil, subprocess, importlib, urllib.request, platform, time

def sh(cmd, t=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)
        return (r.stdout + r.stderr).strip()
    except Exception as e:
        return f"ERR {e}"

print("python", sys.version.split()[0], platform.platform())
print("cpus", os.cpu_count())
print(sh("free -g | head -2"))
print(sh("df -h /content /content/tmp 2>/dev/null | head -4"))
print("ffmpeg:", sh("ffmpeg -version | head -1"))
print("gpu:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "none")

print("\n--- importable packages")
for p in ["torch", "torchaudio", "torchvision", "transformers", "onnxruntime", "mediapipe", "insightface",
          "speechbrain", "sentence_transformers", "lightgbm", "xgboost", "faster_whisper", "ctranslate2",
          "librosa", "soundfile", "av", "decord", "fasttext", "langdetect", "lingua", "feat", "opensmile", "dlib", "face_alignment"]:
    try:
        m = importlib.import_module(p)
        print(f"{p:22s} OK {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"{p:22s} no ({type(e).__name__})")

print("\n--- opencv state")
print(sh("pip list 2>/dev/null | grep -i opencv"))
try:
    import cv2
    print("cv2", cv2.__version__, "Cascade:", hasattr(cv2, "CascadeClassifier"), "FaceDetectorYN:", hasattr(cv2, "FaceDetectorYN"), "dnn:", hasattr(cv2, "dnn"))
except Exception as e:
    print("cv2 import fail", e)

print("\n--- pip dry-run installability (no install)")
for pkg in ["mediapipe", "opencv-python-headless==4.10.0.84", "onnxruntime", "torch", "transformers", "lightgbm",
            "speechbrain", "sentence-transformers", "langdetect", "py-feat", "opensmile"]:
    out = sh(f"pip install --dry-run -q {pkg} 2>&1 | tail -3", 180)
    ok = ("Would install" in out) or ("already satisfied" in out.lower()) or out == ""
    print(f"{pkg:38s} {'OK' if ok else 'FAIL'} | {out[-160:].replace(chr(10), ' ')}")

print("\n--- network")
for u in ["https://huggingface.co/api/models/Systran/faster-whisper-small",
          "https://huggingface.co/api/models/facebook/wav2vec2-base",
          "https://github.com", "https://pypi.org/simple/pip/"]:
    t = time.time()
    try:
        r = urllib.request.urlopen(u, timeout=20)
        print(u, r.status, f"{time.time()-t:.1f}s")
    except Exception as e:
        print(u, "FAIL", type(e).__name__, str(e)[:80])

Writing outputs/claude/src/env_probe.py


In [14]:
apd.run("python outputs/claude/src/env_probe.py")

python 3.13.18 Linux-6.6.159+-x86_64-with-glibc2.39
cpus 2
total        used        free      shared  buff/cache   available
Mem:              12           2           8           0           2          10
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  554M  108G   1% /
overlay         108G  554M  108G   1% /
ffmpeg: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
gpu: /bin/sh: 1: nvidia-smi: not found

--- importable packages
torch                  OK 2.14.1+cpu
torchaudio             OK 2.11.0+cpu
torchvision            OK 0.29.1+cpu
transformers           OK 5.22.1
onnxruntime            OK 1.32.0
mediapipe              no (ModuleNotFoundError)
insightface            no (ModuleNotFoundError)
speechbrain            no (ModuleNotFoundError)
sentence_transformers  OK 5.7.0
lightgbm               OK 4.6.0
xgboost                OK 3.4.1
faster_whisper         OK 1.2.1
ctranslate2            OK 4.9.0
librosa                OK 0.11.0
sou

0

In [15]:
apd.run("pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python; pip install -q mediapipe; pip list 2>/dev/null | grep -i -E 'opencv|mediapipe|numpy|protobuf'")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
mediapipe                             1.2.3
numpy                                 2.4.6
opencv-contrib-python                 5.1.0.95
protobuf                              6.33.6

==================== COPY FROM HERE INTO THE CHAT ====================
$ pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python; pip install -q mediapipe; pip list 2>/dev/null | grep -i -E 'opencv|mediapipe|numpy|protobuf'
[exit code 0, 30s, 6 lines; full log saved as outputs/claude/logs/cmd_0015.log]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not i

0

In [16]:
%%writefile outputs/claude/src/tool_probe.py
"""Phase 2 tool probe: mediapipe FaceLandmarker (frames, speed, blendshape count) and faster-whisper small (speed, language).
Uses one train video; prints only numbers. Never prints transcript text."""
import os, sys, time, subprocess, urllib.request
sys.path.insert(0, "outputs/claude/src/audit")
import numpy as np
import cv2
from common import load_all

print("cv2", cv2.__version__, "Cascade:", hasattr(cv2, "CascadeClassifier"), "FaceDetectorYN:", hasattr(cv2, "FaceDetectorYN"))
import mediapipe as mp
print("mediapipe", mp.__version__)

MD = "outputs/claude/cache/models"
os.makedirs(MD, exist_ok=True)
os.makedirs("/content/tmp", exist_ok=True)
lab, aud, ft = load_all()
row = ft[ft.sid.isin(lab.sid)].iloc[0]
path = row.path
print("probe video (sid hidden), exists:", os.path.exists(path))

# ---- face landmarker
url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
fp = f"{MD}/face_landmarker.task"
try:
    if not os.path.exists(fp):
        t = time.time(); urllib.request.urlretrieve(url, fp); print(f"downloaded landmarker {os.path.getsize(fp)/1e6:.1f} MB in {time.time()-t:.1f}s")
    from mediapipe.tasks import python as mpp
    from mediapipe.tasks.python import vision
    opts = vision.FaceLandmarkerOptions(base_options=mpp.BaseOptions(model_asset_path=fp), output_face_blendshapes=True,
                                        output_facial_transformation_matrixes=True, num_faces=2,
                                        running_mode=vision.RunningMode.IMAGE)
    det = vision.FaceLandmarker.create_from_options(opts)
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print("frames", n)
    for frac in [0.2, 0.4, 0.6, 0.8]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frac * n))
        ok, fr = cap.read()
        if not ok:
            print(frac, "read fail"); continue
        rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        t = time.time()
        res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        dt = time.time() - t
        nf = len(res.face_landmarks)
        nb = len(res.face_blendshapes[0]) if nf else 0
        area = []
        for f in res.face_landmarks:
            xs = [p.x for p in f]; ys = [p.y for p in f]
            area.append(round((max(xs) - min(xs)) * (max(ys) - min(ys)), 3))
        print(f"frac {frac}: faces {nf} blendshapes {nb} face_area_frac {area} matrix {len(res.facial_transformation_matrixes)} {dt*1000:.0f} ms")
except Exception as e:
    print("LANDMARKER FAIL", type(e).__name__, str(e)[:300])

# ---- whisper
try:
    wav = "/content/tmp/probe.wav"
    subprocess.run(f'ffmpeg -y -loglevel error -ss 120 -t 60 -i "{path}" -vn -ac 1 -ar 16000 {wav}', shell=True, check=True)
    import faster_whisper
    from faster_whisper import WhisperModel
    print("faster_whisper", faster_whisper.__version__)
    t = time.time()
    model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=2, download_root=f"{MD}/whisper")
    print(f"model load {time.time()-t:.1f}s")
    t = time.time()
    segs, info = model.transcribe(wav, beam_size=1, vad_filter=True, word_timestamps=False)
    segs = list(segs)
    dt = time.time() - t
    words = sum(len(s.text.split()) for s in segs)
    print(f"60 s clip: {dt:.1f}s wall (RTF {dt/60:.2f}), lang {info.language} p={info.language_probability:.2f}, segments {len(segs)}, words {words}")
    os.remove(wav)
except Exception as e:
    print("WHISPER FAIL", type(e).__name__, str(e)[:300])

Writing outputs/claude/src/tool_probe.py


In [17]:
apd.run("python outputs/claude/src/tool_probe.py")

I0000 00:00:1795073547.649510    5718 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795073549.263583    5718 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795073555.130302    5718 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
W0000 00:00:1795073559.206602    5810 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795073559.234441    5814 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795073559.285751    5815 inference_feedback_manager.cc:121] Feedback manager

0

In [18]:
%%writefile outputs/claude/src/whisper_probe.py
"""Phase 2 probe 2: faster-whisper small with numpy-array input (bypasses faster_whisper.decode_audio / PyAV 19 incompat).
Prints timing, language, word counts only. No transcript text."""
import os, sys, time, subprocess
sys.path.insert(0, "outputs/claude/src/audit")
import numpy as np, soundfile as sf
from common import load_all
from faster_whisper import WhisperModel

MD = "outputs/claude/cache/models"
os.makedirs("/content/tmp", exist_ok=True)
lab, aud, ft = load_all()
tr = ft[ft.sid.isin(lab.sid)].reset_index(drop=True)

def clip(path, ss, dur, out="/content/tmp/probe.wav"):
    subprocess.run(f'ffmpeg -y -loglevel error -ss {ss} -t {dur} -i "{path}" -vn -ac 1 -ar 16000 {out}', shell=True, check=True)
    x, sr = sf.read(out, dtype="float32")
    os.remove(out)
    return x

model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=2, download_root=f"{MD}/whisper")
for k, idx in enumerate([0, 40, 90]):
    p = tr.path.iloc[idx]
    dur = 60 if k == 0 else 30
    x = clip(p, 120, dur)
    t = time.time()
    segs, info = model.transcribe(x, beam_size=1, vad_filter=True)
    segs = list(segs)
    dt = time.time() - t
    words = sum(len(s.text.split()) for s in segs)
    print(f"video#{idx}: {len(x)/16000:.0f}s audio, {dt:.1f}s wall (RTF {dt/(len(x)/16000):.2f}), lang {info.language} p={info.language_probability:.2f}, segments {len(segs)}, words {words}")

Writing outputs/claude/src/whisper_probe.py


In [19]:
apd.run("python outputs/claude/src/whisper_probe.py 2>&1 | grep -v -E '^(I0000|W0000|WARNING)'")

Traceback (most recent call last):
  File "/content/research/outputs/claude/src/whisper_probe.py", line 20, in <module>
    model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=2, download_root=f"{MD}/whisper")
  File "/usr/local/lib/python3.13/dist-packages/faster_whisper/transcribe.py", line 689, in __init__
    self.model = ctranslate2.models.Whisper(
                 ~~~~~~~~~~~~~~~~~~~~~~~~~~^
        model_path,
        ^^^^^^^^^^^
    ...<6 lines>...
        **model_kwargs,
        ^^^^^^^^^^^^^^^
    )
    ^
RuntimeError: Unsupported model binary version. This executable supports models with binary version v6 or below, but the model has binary version v774843950. This usually means that the model was generated by a later version of CTranslate2. (Forward compatibility is not guaranteed.)

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/whisper_probe.py 2>&1 | grep -v -E '^(I0000|W0000|WARNING)'
[exit code

0

In [20]:
apd.run("ls -laR outputs/claude/cache/models/whisper | cut -c1-160 | head -40; echo '--- head bytes of model.bin'; for f in $(find outputs/claude/cache/models/whisper -name model.bin); do ls -laL $f; head -c 64 $f | xxd | head -3; done; pip list 2>/dev/null | grep -i -E 'ctranslate2|faster|huggingface'")

outputs/claude/cache/models/whisper:
total 21
drwx------ 5 root root 4096 Nov 19 07:32 .
drwx------ 3 root root 4096 Nov 19 07:32 ..
drwx------ 3 root root 4096 Nov 19 07:32 blobs
-rw------- 1 root root  191 Nov 19 07:32 CACHEDIR.TAG
drwx------ 3 root root 4096 Nov 19 07:32 .locks
drwx------ 6 root root 4096 Nov 19 07:32 models--Systran--faster-whisper-small

outputs/claude/cache/models/whisper/blobs:
total 13
drwx------ 3 root root 4096 Nov 19 07:32 .
drwx------ 5 root root 4096 Nov 19 07:32 ..
drwx------ 2 root root 4096 Nov 19 07:32 42
-r-------- 1 root root    2 Nov 19 07:32 .huggingface-shared-blobs

outputs/claude/cache/models/whisper/blobs/42:
total 472223
drwx------ 2 root root      4096 Nov 19 07:32 .
drwx------ 3 root root      4096 Nov 19 07:32 ..
-rw------- 1 root root 483546902 Nov 19 07:32 429ffc84492c1122da005d7274fc397bf142fe12066eb7e4612dc25c579db2ea
-rw------- 1 root root         0 Nov 19 07:32 429ffc84492c1122da005d7274fc397bf142fe12066eb7e4612dc25c579db2ea.lock
-rw-

0

In [21]:
%%writefile outputs/claude/src/fetch_whisper.py
"""Materialise faster-whisper-small as plain files (no symlinks) in cache/models/whisper_small, verify sizes, then probe speed.
Prints only sizes, timings, language, word counts."""
import os, sys, time, shutil, subprocess, glob
sys.path.insert(0, "outputs/claude/src/audit")
import numpy as np, soundfile as sf

MD = "outputs/claude/cache/models"
DST = f"{MD}/whisper_small"
LOCAL = "/content/whisper_small"
REPO = "Systran/faster-whisper-small"
need = ["model.bin", "config.json", "tokenizer.json", "vocabulary.txt", "preprocessor_config.json"]


def ok(d):
    try:
        return all(os.path.exists(f"{d}/{f}") for f in need[:2]) and os.path.getsize(f"{d}/model.bin") > 4e8
    except Exception:
        return False


if not ok(DST):
    from huggingface_hub import hf_hub_download
    os.makedirs(LOCAL, exist_ok=True)
    for f in need:
        try:
            p = hf_hub_download(REPO, f, local_dir=LOCAL)
            print(f"{f}: {os.path.getsize(p)/1e6:.2f} MB")
        except Exception as e:
            print(f"{f}: skip ({type(e).__name__})")
    assert os.path.getsize(f"{LOCAL}/model.bin") > 4e8, "model.bin too small"
    os.makedirs(DST, exist_ok=True)
    for f in need:
        if os.path.exists(f"{LOCAL}/{f}"):
            shutil.copyfile(f"{LOCAL}/{f}", f"{DST}/{f}")
    for f in need:
        if os.path.exists(f"{LOCAL}/{f}"):
            assert os.path.getsize(f"{DST}/{f}") == os.path.getsize(f"{LOCAL}/{f}"), f"size mismatch {f}"
    shutil.rmtree(f"{MD}/whisper", ignore_errors=True)  # remove broken symlink-stub cache
print("model dir ok:", ok(DST), "model.bin MB", round(os.path.getsize(f"{DST}/model.bin")/1e6, 1))

from faster_whisper import WhisperModel
from common import load_all
lab, aud, ft = load_all()
tr = ft[ft.sid.isin(lab.sid)].reset_index(drop=True)
os.makedirs("/content/tmp", exist_ok=True)
t = time.time()
model = WhisperModel(LOCAL if ok(LOCAL) else DST, device="cpu", compute_type="int8", cpu_threads=2)
print(f"model load {time.time()-t:.1f}s")
for k, idx in enumerate([0, 40, 90]):
    p = tr.path.iloc[idx]
    dur = 60 if k == 0 else 30
    out = "/content/tmp/probe.wav"
    subprocess.run(f'ffmpeg -y -loglevel error -ss 120 -t {dur} -i "{p}" -vn -ac 1 -ar 16000 {out}', shell=True, check=True)
    x, sr = sf.read(out, dtype="float32"); os.remove(out)
    t = time.time()
    segs, info = model.transcribe(x, beam_size=1, vad_filter=True)
    segs = list(segs); dt = time.time() - t
    words = sum(len(s.text.split()) for s in segs)
    print(f"video#{idx}: {len(x)/16000:.0f}s audio, {dt:.1f}s wall (RTF {dt/(len(x)/16000):.2f}), lang {info.language} p={info.language_probability:.2f}, segments {len(segs)}, words {words}")

Writing outputs/claude/src/fetch_whisper.py


In [22]:
apd.run("python outputs/claude/src/fetch_whisper.py 2>&1 | grep -v -E '^(I0000|W0000|WARNING)'")

model.bin: 483.55 MB
config.json: 0.00 MB
tokenizer.json: 2.20 MB
vocabulary.txt: 0.46 MB
preprocessor_config.json: skip (RemoteEntryNotFoundError)
model dir ok: True model.bin MB 483.5
model load 2.0s
video#0: 60s audio, 40.6s wall (RTF 0.68), lang ru p=0.93, segments 24, words 152
video#40: 30s audio, 21.2s wall (RTF 0.71), lang ru p=0.99, segments 9, words 54
video#90: 30s audio, 25.0s wall (RTF 0.83), lang ru p=0.99, segments 5, words 55

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/fetch_whisper.py 2>&1 | grep -v -E '^(I0000|W0000|WARNING)'
[exit code 0, 118s, 11 lines; full log saved as outputs/claude/logs/cmd_0019.log]
model.bin: 483.55 MB
config.json: 0.00 MB
tokenizer.json: 2.20 MB
vocabulary.txt: 0.46 MB
preprocessor_config.json: skip (RemoteEntryNotFoundError)
model dir ok: True model.bin MB 483.5
model load 2.0s
video#0: 60s audio, 40.6s wall (RTF 0.68), lang ru p=0.93, segments 24, words 152
video#40: 30s audio, 21.2s w

0

In [23]:
%%writefile outputs/claude/proposal.md
# Proposal (Phase 2) — committed before building

Construct: acute psychological distress (APD) risk. Cohorts: elevated-risk (n=80 train) vs control (n=66 train). 62 unlabelled test sessions.

## Facts that shape the design
- CPU only (2 cores, 12 GB RAM); no GPU. Whisper-small int8 runs at real-time factor 0.68-0.83; spoken language so far Russian (p >= 0.93 on 3 probe clips), but mixed language is expected, so it is detected per window.
- Full-length processing (~32 h of video) is infeasible; interview duration is itself confounded with cohort (duration-only AUROC ~0.64, all audit columns ~0.65, permutation p=0.016). Luminance and SNR are not meaningfully confounded.
- Therefore every session is represented by the SAME amount of material: 3 windows x 45 s, centred at 20%, 50%, 80% of the session (start clipped to [0, duration-45]; sessions shorter than 135 s use overlapping windows, flagged). Duration, resolution, fps and every audit.csv column are NEVER model inputs.
- n=146 training sessions: only strongly regularised, low-capacity models; no deep fine-tuning.

## Channels (each produces a 0-100 score)
1. Vision. MediaPipe FaceLandmarker (mediapipe 1.2.3, face_landmarker.task float16 v1), 5 frames/s inside the windows, largest face is the participant. Per frame: 52 blendshape coefficients, head pose from the facial transformation matrix (yaw/pitch/roll), gaze and blink blendshapes. Per window and session: mean, SD, 10th and 90th percentile of blendshapes; head-pose SD and mean absolute velocity; blink rate; face-detect rate (a quality covariate, not a model input). Features pooled across windows by the mean.
2. Language. Speech recognition with faster-whisper 1.2.1 / ctranslate2 4.9.0, model Systran/faster-whisper-small (int8, beam 1, VAD on), language detected per window. Features: multilingual sentence embedding (intfloat/multilingual-e5-small via sentence-transformers 5.7.0; mean over window segments) reduced by PCA fitted inside each training fold; plus language-independent speech stats (words/min, mean word length, type-token ratio, pause count and mean pause length from segment gaps, segment length SD). Transcript text is NOT stored on disk; only embeddings and counts.
3. Acoustic. openSMILE eGeMAPSv02 functionals (88 features) per window, averaged, plus speech fraction inside windows. Vocal acoustics are independent of the ASR.

## Classifier and fusion
- Per channel: z-scored features -> L2 logistic regression (strong regularisation, C chosen by inner 3-fold CV inside each outer training fold). LightGBM depth-2 is a secondary comparison only; the simpler model is preferred unless the gain exceeds CV noise (~0.05).
- Fusion: late fusion; out-of-fold channel scores stacked by logistic regression (nested, so the stacker never sees its own training scores), compared with early fusion (concatenated features) and with the best single channel. Final risk_score = calibrated probability x 100 (Platt, fitted on out-of-fold scores); risk_label = elevated if score >= 50 (train prevalence 55%; threshold reviewed in Phase 4).
- Test predictions: all models refit on all 146 training sessions; fixed seed 0.

## Validation
- Stratified 5-fold CV grouped by participant_id (each participant has one session, so grouping = stratification), repeated over 5 seeds; report mean and SD; the saved train_cv_predictions.csv uses seed 0.
- Mandatory confound reporting next to every headline number: (a) duration-only floor 0.64; (b) AUROC of the model score after regressing out log-duration (residualised) ; (c) correlation of the model score with log-duration; (d) AUROC within duration, SNR and luminance tertiles (duration/SNR from audit.csv; luminance tertile is our stand-in, see decisions.md). A channel is only credited with signal if it beats the floor and holds within the duration tertiles.
- Permutation test (participant label shuffles) for the final fused model.
- Test labels are never used or inferred; test features are computed by the identical code path.

## Affect analysis (Phase 5)
Constructs are defined a priori from the features, per channel, and each is tested against cohort with a duration-adjusted logistic model and a participant-level permutation test; correction across constructs (Holm):
- sadness: brow-inner-up, mouth-frown, lowered mouth corners (vision); low pitch and energy (acoustic).
- flatness: low blendshape variance, low head-pose variance (vision); low f0 range and loudness variability (acoustic).
- anxiety: blink rate, head-pose velocity, gaze variability (vision); speech rate and pitch variability (acoustic).
- withdrawal: gaze away, head pitch down (vision); low words/min, long pauses, low speech fraction (language/acoustic).
- mixed states: pairwise co-occurrence of the above, plus a 2-component clustering on vision features inspected for cohort composition. Labels in channels.csv affect_labels come from per-session z-scores against the control training distribution (>1 SD).
These are descriptive associations on 146 sessions, not diagnoses.

## Compute plan (wall-clock, counted conservatively as 2 CPU-hours per wall hour)
Whisper ~135 s audio x RTF 0.75 x 208 ~ 5.9 h; vision 675 frames/session ~45 s x 208 ~ 2.6 h; openSMILE and embeddings ~0.5 h; training/CV < 0.5 h. Total ~9.5 h wall ~ 19 CPU-h, under the 24 CPU-h cap. If cumulative wall time passes 11 h before all sessions are done, language falls back to Whisper-base for the remaining sessions and the switch is logged. No GPU needed.

## Known risks
- Small n: CV AUROC SD of ~0.03-0.05; model differences below ~0.05 are not interpretable.
- Whisper-small Russian/Kazakh accuracy is imperfect; the language channel may mostly capture speech-rate and pause behaviour.
- Cohort differences in session length may reflect cohort differences in interview content and thus also appear in windowed features; the residualised and stratified analyses limit but cannot remove this.
- Six extreme train sessions (all elevated-risk): handled by the uniform windows; the near-silent audio one (-85 dB) will likely fail ASR and is recorded in failures.csv.

Writing outputs/claude/proposal.md


In [24]:
%%writefile outputs/claude/src/phase2_close.py
"""Phase 2 close-out: pin requirements, append decisions, update STATE.md."""
OUT = "outputs/claude"
open(f"{OUT}/requirements.txt", "w").write("""numpy
pandas
scipy
scikit-learn
lightgbm==4.6.0
librosa==0.11.0
soundfile==0.14.0
faster-whisper==1.2.1
ctranslate2==4.9.0
mediapipe==1.2.3
opencv-contrib-python==5.1.0.95
sentence-transformers==5.7.0
transformers==5.22.1
huggingface_hub
opensmile
""")
dec = """
## 2026-11-19 Phase 2 (proposal)
- Environment: CPU only (2 cores, 12 GB), Python 3.13, no GPU. Network (HF, GitHub, PyPI) reachable.
- OpenCV: Colab had opencv-python 5.0 + opencv-contrib 4.14 (no CascadeClassifier). Uninstalled all three and installed mediapipe, which pulled opencv-contrib-python 5.1.0.95 (Cascade present). requirements.txt pins it; run.sh must install requirements before anything else. The unrelated dopamine-rl pip warning is ignored.
- Visual channel: MediaPipe FaceLandmarker 52 blendshapes + head pose chosen over heavier backbones: 20-90 ms/frame on CPU, one face found in all 4 probe frames.
- faster-whisper decode_audio fails with PyAV 19 (av.open metadata_errors). Workaround: decode with ffmpeg to 16 kHz mono float32 numpy and pass the array to transcribe().
- Whisper model cache on Drive broke: HF snapshot symlinks became 79-byte text stubs (loader reported a nonsense binary version v774843950). Workaround: download with local_dir to plain files and keep a copy at cache/models/whisper_small (model.bin 483.5 MB, size-verified); load by path. Do not use download_root on Drive.
- Whisper small int8, 2 threads: RTF 0.68-0.83; probe clips detected as Russian (p 0.93-0.99).
- DECISION: windowed representation, 3 x 45 s per session at 20/50/80% of duration, identical for every session, because full processing is infeasible (~32 h video) and duration is confounded with cohort (~0.64 AUROC). No audit.csv column and no duration is a model input.
- DECISION: methodology committed in proposal.md (logistic regression per channel on vision, language, openSMILE acoustics; nested late fusion). Revisions will be logged here.
- DECISION: transcript text is not persisted; only embeddings and counts are cached.
"""
open(f"{OUT}/decisions.md", "a").write(dec)
state = """phase: 2 COMPLETE -> entering phase 3 (build)
done:
- Phase 1 audit (see audit_notes.md); Phase 2 environment probe + proposal.md + decisions.md + pinned requirements.txt
- Working tools verified: mediapipe 1.2.3 FaceLandmarker (cache/models/face_landmarker.task), faster-whisper small (cache/models/whisper_small, plain files), opencv-contrib 5.1.0.95
in progress: nothing
next command: apd.run("pip install -q -r outputs/claude/requirements.txt") then write src/common.py (windows, paths, cache helpers) and src/extract_session.py (vision + audio + ASR per session, cached per sid, --limit 3 first)
open issues: opensmile not yet installed/verified on py3.13; multilingual-e5-small download untested; luminance tertile is a stand-in; duration confound (~0.64); extreme sessions flagged in audit_notes.md
"""
open(f"{OUT}/STATE.md", "w").write(state)
print("phase 2 close-out written")

Writing outputs/claude/src/phase2_close.py


In [25]:
apd.run("python outputs/claude/src/phase2_close.py")

phase 2 close-out written

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/phase2_close.py
[exit code 0, 0s, 1 lines; full log saved as outputs/claude/logs/cmd_0020.log]
phase 2 close-out written
==================== END OF COPY ====================


0

In [26]:
apd.ls()

==================== COPY FROM HERE INTO THE CHAT ====================
STATE.md  806 B
audit_notes.md  2,799 B
decisions.md  3,530 B
operator_notes.md  615 B
proposal.md  6,115 B
requirements.txt  248 B
cache/  (11 files, not listed)
logs/  (23 files, not listed)
src/env_probe.py  2,573 B
src/fetch_whisper.py  2,686 B
src/phase2_close.py  3,102 B
src/tool_probe.py  3,544 B
src/whisper_probe.py  1,400 B
src/audit/common.py  1,877 B
src/audit/finish_audit.py  9,710 B
src/audit/inventory.py  9,161 B
src/audit/luminance.py  1,744 B
src/audit/__pycache__/common.cpython-313.pyc  4,152 B
==================== END OF COPY ====================


## phase 3

In [27]:
apd.run("pip install -q -r outputs/claude/requirements.txt 2>&1 | tail -n 5; pip list 2>/dev/null | grep -i -E 'opensmile|mediapipe|opencv|sentence|ctranslate2|faster'")

ctranslate2                           4.9.0
faster-whisper                        1.2.1
mediapipe                             1.2.3
opencv-contrib-python                 5.1.0.95
opensmile                             2.6.0
sentence-transformers                 5.7.0
sentencepiece                         0.2.2

==================== COPY FROM HERE INTO THE CHAT ====================
$ pip install -q -r outputs/claude/requirements.txt 2>&1 | tail -n 5; pip list 2>/dev/null | grep -i -E 'opensmile|mediapipe|opencv|sentence|ctranslate2|faster'
[exit code 0, 9s, 7 lines; full log saved as outputs/claude/logs/cmd_0021.log]
ctranslate2                           4.9.0
faster-whisper                        1.2.1
mediapipe                             1.2.3
opencv-contrib-python                 5.1.0.95
opensmile                             2.6.0
sentence-transformers                 5.7.0
sentencepiece                         0.2.2
==================== END OF COPY ====================


0

In [28]:
%%writefile outputs/claude/src/verify_tools.py
"""Phase 3 tool check: openSMILE eGeMAPSv02 on a 45 s clip, multilingual-e5-small as plain files, encode a dummy sentence.
Prints counts and timings only."""
import os, sys, time, shutil, subprocess
sys.path.insert(0, "outputs/claude/src/audit")
import numpy as np, soundfile as sf
from common import load_all

MD = "outputs/claude/cache/models"
os.makedirs("/content/tmp", exist_ok=True)

# ---- openSMILE
try:
    import opensmile
    print("opensmile", opensmile.__version__)
    lab, aud, ft = load_all()
    p = ft[ft.sid.isin(lab.sid)].path.iloc[0]
    out = "/content/tmp/v.wav"
    subprocess.run(f'ffmpeg -y -loglevel error -ss 100 -t 45 -i "{p}" -vn -ac 1 -ar 16000 {out}', shell=True, check=True)
    x, sr = sf.read(out, dtype="float32"); os.remove(out)
    sm = opensmile.Smile(feature_set=opensmile.FeatureSet.eGeMAPSv02, feature_level=opensmile.FeatureLevel.Functionals)
    t = time.time()
    f = sm.process_signal(x, sr)
    print(f"eGeMAPSv02: {f.shape[1]} features, {time.time()-t:.1f}s, nan {int(f.isna().sum().sum())}")
except Exception as e:
    print("OPENSMILE FAIL", type(e).__name__, str(e)[:300])

# ---- e5 encoder
try:
    DST = f"{MD}/e5_small"
    LOCAL = "/content/e5_small"
    pats = ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json", "sentencepiece.bpe.model",
            "special_tokens_map.json", "modules.json", "sentence_bert_config.json", "1_Pooling/config.json"]
    if not os.path.exists(f"{DST}/model.safetensors"):
        from huggingface_hub import snapshot_download
        snapshot_download("intfloat/multilingual-e5-small", local_dir=LOCAL, allow_patterns=pats)
        shutil.copytree(LOCAL, DST, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
        for r, _, fs in os.walk(LOCAL):
            for fn in fs:
                a = os.path.join(r, fn)
                if ".cache" in a: continue
                b = a.replace(LOCAL, DST)
                assert os.path.getsize(a) == os.path.getsize(b), f"size mismatch {fn}"
    print("e5 files:", sorted(os.listdir(DST)), "safetensors MB", round(os.path.getsize(f"{DST}/model.safetensors")/1e6, 1))
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(LOCAL if os.path.exists(LOCAL) else DST, device="cpu")
    t = time.time()
    e = m.encode(["query: Привет, как дела", "query: hello how are you"], normalize_embeddings=True)
    print("e5 embedding", e.shape, f"{time.time()-t:.2f}s", "cos(ru,en)", round(float(e[0] @ e[1]), 3))
except Exception as e:
    print("E5 FAIL", type(e).__name__, str(e)[:300])

Writing outputs/claude/src/verify_tools.py


In [29]:
apd.run("python outputs/claude/src/verify_tools.py 2>&1 | grep -v -E '^(I0000|W0000|WARNING)'")

opensmile 2.6.0
eGeMAPSv02: 88 features, 6.2s, nan 0

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 9 files: 100%|██████████| 9/9 [00:08<00:00,  1.02it/s]
e5 files: ['1_Pooling', 'config.json', 'model.safetensors', 'modules.json', 'sentence_bert_config.json', 'sentencepiece.bpe.model', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json'] safetensors MB 470.6

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2258.62it/s]
e5 embedding (2, 384) 0.19s cos(ru,en) 0.871

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/verify_tools.py 2>&1 | grep -v -E '^(I0000|W0000|WARNING)'
[exit code 0, 127s, 14 lines; full log saved as outputs/claude/logs/cmd_0022.log]
opensmile 2.6.0
eGeMAPSv02: 88 features, 6.2s, nan 0

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

0

In [30]:
%%writefile outputs/claude/src/extract_session.py
"""Per-session feature extraction (label-blind). Stages: vision (MediaPipe FaceLandmarker), acoustic (openSMILE eGeMAPSv02),
asr (faster-whisper small -> language stats + multilingual-e5-small embedding). 3 x 45 s windows at 20/50/80 % of duration.
Each stage is cached per session under outputs/claude/cache/feat/<stage>/. Transcript text is never saved or printed."""
import os, sys, json, time, argparse, subprocess, shutil, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "audit"))
from common import load_all

SEED = 0
OUT = "outputs/claude"
CACHE = f"{OUT}/cache"
FEAT = f"{CACHE}/feat"
MODELS = f"{CACHE}/models"
WIN_S = 45.0
FRACS = (0.2, 0.5, 0.8)
VIS_FPS = 5.0
STAGES = ["vision", "acoustic", "asr"]
np.random.seed(SEED)


def windows(dur):
    starts = []
    for f in FRACS:
        s = f * dur - WIN_S / 2
        starts.append(round(min(max(s, 0.0), max(0.0, dur - WIN_S)), 2))
    return starts, bool(dur < WIN_S * len(FRACS))


def atomic_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


def local_model(name):
    src, dst = f"{MODELS}/{name}", f"/content/{name}"
    if not os.path.exists(f"{dst}/.copied"):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/.copied", "w").write("ok")
    return dst


def read_audio(path, start, dur=WIN_S):
    cmd = ["ffmpeg", "-loglevel", "error", "-ss", str(start), "-t", str(dur), "-i", path,
           "-vn", "-ac", "1", "-ar", "16000", "-f", "f32le", "-"]
    r = subprocess.run(cmd, capture_output=True, check=True)
    return np.frombuffer(r.stdout, dtype=np.float32).copy()


# ------------------------------------------------------------------ vision
def vision_stage(path, starts, landmarker):
    import cv2
    import mediapipe as mp
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    stride = max(1, int(round(fps / VIS_FPS)))
    names = None
    B, P, T, W = [], [], [], []
    n_proc = n_det = n_multi = 0
    areas = []
    for wi, s in enumerate(starts):
        cap.set(cv2.CAP_PROP_POS_MSEC, s * 1000.0)
        for k in range(int(WIN_S * fps)):
            if not cap.grab():
                break
            if k % stride:
                continue
            ok, fr = cap.retrieve()
            if not ok:
                continue
            n_proc += 1
            res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
            nf = len(res.face_landmarks)
            if nf == 0:
                continue
            n_det += 1
            n_multi += int(nf > 1)
            ar = []
            for f in res.face_landmarks:
                xs = [p.x for p in f]; ys = [p.y for p in f]
                ar.append((max(xs) - min(xs)) * (max(ys) - min(ys)))
            i = int(np.argmax(ar))
            areas.append(ar[i])
            cats = res.face_blendshapes[i]
            if names is None:
                names = [c.category_name for c in cats]
            B.append([c.score for c in cats])
            R = np.asarray(res.facial_transformation_matrixes[i])[:3, :3]
            R = R / np.linalg.norm(R, axis=0, keepdims=True)
            sy = np.sqrt(R[0, 0] ** 2 + R[1, 0] ** 2)
            P.append([np.degrees(np.arctan2(-R[2, 0], sy)), np.degrees(np.arctan2(R[2, 1], R[2, 2])),
                      np.degrees(np.arctan2(R[1, 0], R[0, 0]))])  # yaw, pitch, roll (axis convention not validated)
            T.append(s + k / fps)
            W.append(wi)
    cap.release()
    if n_det < 30:
        raise RuntimeError(f"too few face frames ({n_det}/{n_proc})")
    B, P, T, W = np.array(B), np.array(P), np.array(T), np.array(W)
    f = {}
    for j, nm in enumerate(names):
        if nm == "_neutral":
            continue
        col = B[:, j]
        f[f"v_{nm}_mean"] = float(col.mean()); f[f"v_{nm}_sd"] = float(col.std())
        f[f"v_{nm}_p10"] = float(np.percentile(col, 10)); f[f"v_{nm}_p90"] = float(np.percentile(col, 90))
    ix = {n: j for j, n in enumerate(names)}
    g = lambda n: B[:, ix[n]]
    gh = ((g("eyeLookOutLeft") + g("eyeLookInRight")) - (g("eyeLookInLeft") + g("eyeLookOutRight"))) / 2
    gv = (g("eyeLookUpLeft") + g("eyeLookUpRight") - g("eyeLookDownLeft") - g("eyeLookDownRight")) / 2
    f.update(v_gaze_h_absmean=float(np.abs(gh).mean()), v_gaze_h_sd=float(gh.std()),
             v_gaze_v_mean=float(gv.mean()), v_gaze_v_sd=float(gv.std()))
    for k, nm in enumerate(["yaw", "pitch", "roll"]):
        f[f"v_{nm}_mean"] = float(P[:, k].mean()); f[f"v_{nm}_sd"] = float(P[:, k].std())
    dt = np.diff(T); same = (np.diff(W) == 0) & (dt > 0) & (dt <= 1.5 * stride / fps)
    if same.any():
        vel = np.abs(np.diff(P, axis=0))[same].mean(axis=1) / dt[same]
        f["v_pose_vel"] = float(vel.mean())
        b = (g("eyeBlinkLeft") + g("eyeBlinkRight")) / 2
        rise = (b[:-1] < 0.5) & (b[1:] >= 0.5) & same
        f["v_blink_rate_per_min"] = float(rise.sum() / dt[same].sum() * 60.0)
    else:
        f["v_pose_vel"] = np.nan; f["v_blink_rate_per_min"] = np.nan
    f["v_blink_frac"] = float((((g("eyeBlinkLeft") + g("eyeBlinkRight")) / 2) >= 0.5).mean())
    f["qc_face_detect_rate"] = n_det / max(n_proc, 1)
    f["qc_multi_face_frac"] = n_multi / max(n_det, 1)
    f["qc_face_area_mean"] = float(np.mean(areas))
    f["qc_vis_frames"] = int(n_det)
    return f, {"fps": fps, "stride": stride, "n_proc": n_proc, "n_det": n_det}


# ------------------------------------------------------------------ acoustic
def acoustic_stage(audio, smile):
    rows = []
    for x in audio:
        if len(x) < 16000 * 3:
            continue
        rows.append(smile.process_signal(x, 16000).iloc[0])
    if not rows:
        raise RuntimeError("no usable audio window")
    M = np.array([r.values for r in rows], dtype=float)
    cols = list(rows[0].index)
    m = np.nanmean(M, axis=0)
    return {f"a_{c}": float(v) for c, v in zip(cols, m)}, {"n_windows": len(rows)}


# ------------------------------------------------------------------ asr
def asr_stage(audio, whisper, e5):
    per, embs, langs, lps = [], [], [], []
    for x in audio:
        wl = len(x) / 16000
        if wl < 3:
            langs.append("na"); continue
        segs, info = whisper.transcribe(x, beam_size=1, vad_filter=True)
        segs = list(segs)
        langs.append(f"{info.language}:{info.language_probability:.2f}")
        words = [w for s in segs for w in s.text.split()]
        if segs:
            lps.append(np.mean([s.avg_logprob for s in segs]))
        st, en = np.array([s.start for s in segs]), np.array([s.end for s in segs])
        gaps = (st[1:] - en[:-1]) if len(segs) > 1 else np.array([])
        gaps = gaps[gaps > 0.3]
        lw = [w.lower().strip(".,!?;:\"'()-") for w in words]
        d = dict(l_words_per_min=len(words) / (wl / 60.0),
                 l_mean_word_len=float(np.mean([len(w) for w in words])) if words else np.nan,
                 l_ttr=len(set(lw)) / len(lw) if len(lw) >= 5 else np.nan,
                 l_pause_count=float(len(gaps)), l_pause_mean=float(gaps.mean()) if len(gaps) else 0.0,
                 l_seg_len_sd=float(np.std(en - st)) if len(segs) > 1 else np.nan,
                 l_speech_frac=float((en - st).sum() / wl) if len(segs) else 0.0,
                 l_n_segments=float(len(segs)))
        per.append(d)
        if len(words) >= 3:
            text = "query: " + " ".join(s.text.strip() for s in segs)
            embs.append(e5.encode([text], normalize_embeddings=True)[0])
    if not per:
        raise RuntimeError("no usable audio window")
    f = {k: float(np.nanmean([d[k] for d in per])) if not all(np.isnan(d[k]) for d in per) else np.nan for k in per[0]}
    f["qc_asr_logprob_mean"] = float(np.mean(lps)) if lps else np.nan
    f["qc_asr_windows_with_text"] = int(len(embs))
    emb = np.mean(embs, axis=0) if embs else np.full(384, np.nan)
    return f, emb, {"langs": langs}


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0, help="max sessions needing work to process (0 = all)")
    ap.add_argument("--split", default="all", choices=["all", "train", "test"])
    ap.add_argument("--sids", default="")
    ap.add_argument("--stages", default=",".join(STAGES))
    ap.add_argument("--retry-failed", action="store_true")
    a = ap.parse_args()
    stages = [s for s in a.stages.split(",") if s]
    for s in stages + ["fail"]:
        os.makedirs(f"{FEAT}/{s}", exist_ok=True)

    lab, aud, ft = load_all()
    mg = pd.read_csv(f"{CACHE}/audit/merged.csv")[["sid", "split", "duration_s"]]
    df = ft[["sid", "path"]].merge(mg, on="sid").sort_values("sid").reset_index(drop=True)
    if a.split != "all":
        df = df[df.split == a.split]
    if a.sids:
        df = df[df.sid.isin(a.sids.split(","))]

    def todo(sid):
        out = []
        for s in stages:
            ext = "npy" if False else "json"
            if os.path.exists(f"{FEAT}/{s}/{sid}.json"):
                continue
            if os.path.exists(f"{FEAT}/fail/{sid}__{s}.json") and not a.retry_failed:
                continue
            out.append(s)
        return out

    models = {}
    n_done = 0
    t_start = time.time()
    for _, r in df.iterrows():
        need = todo(r.sid)
        if not need:
            continue
        if a.limit and n_done >= a.limit:
            break
        starts, short = windows(float(r.duration_s))
        line = [f"{n_done+1:3d} {r.sid} dur {r.duration_s:.0f}s{' SHORT' if short else ''}"]
        audio = None
        if any(s in need for s in ("acoustic", "asr")):
            try:
                audio = [read_audio(r.path, s) for s in starts]
            except Exception as e:
                for s in ("acoustic", "asr"):
                    if s in need:
                        atomic_json(f"{FEAT}/fail/{r.sid}__{s}.json", {"stage": s, "reason": f"audio read: {type(e).__name__} {str(e)[:150]}"})
                need = [s for s in need if s not in ("acoustic", "asr")]
                line.append("AUDIO-FAIL")
        for s in need:
            t0 = time.time()
            try:
                if s == "vision":
                    if "lm" not in models:
                        import mediapipe as mp
                        from mediapipe.tasks import python as mpp
                        from mediapipe.tasks.python import vision as mpv
                        opts = mpv.FaceLandmarkerOptions(
                            base_options=mpp.BaseOptions(model_asset_path=f"{MODELS}/face_landmarker.task"),
                            output_face_blendshapes=True, output_facial_transformation_matrixes=True,
                            num_faces=2, running_mode=mpv.RunningMode.IMAGE)
                        models["lm"] = mpv.FaceLandmarker.create_from_options(opts)
                    f, meta = vision_stage(r.path, starts, models["lm"])
                    note = f"det {f['qc_face_detect_rate']:.2f} multi {f['qc_multi_face_frac']:.2f}"
                elif s == "acoustic":
                    if "smile" not in models:
                        import opensmile
                        models["smile"] = opensmile.Smile(feature_set=opensmile.FeatureSet.eGeMAPSv02,
                                                          feature_level=opensmile.FeatureLevel.Functionals)
                    f, meta = acoustic_stage(audio, models["smile"])
                    note = f"nan {int(np.isnan(list(f.values())).sum())}"
                else:
                    if "wh" not in models:
                        from faster_whisper import WhisperModel
                        from sentence_transformers import SentenceTransformer
                        models["wh"] = WhisperModel(local_model("whisper_small"), device="cpu", compute_type="int8", cpu_threads=2)
                        models["e5"] = SentenceTransformer(local_model("e5_small"), device="cpu")
                    f, emb, meta = asr_stage(audio, models["wh"], models["e5"])
                    np.save(f"{FEAT}/asr/{r.sid}_emb.npy", emb)
                    note = f"lang {','.join(meta['langs'])} wpm {f['l_words_per_min']:.0f}"
                wall = time.time() - t0
                atomic_json(f"{FEAT}/{s}/{r.sid}.json", {"sid": r.sid, "stage": s, "wall_s": wall, "starts": starts,
                                                        "short_session": short, "meta": meta, "features": f})
                line.append(f"{s} {wall:.0f}s [{note}]")
            except Exception as e:
                atomic_json(f"{FEAT}/fail/{r.sid}__{s}.json", {"stage": s, "reason": f"{type(e).__name__} {str(e)[:200]}"})
                line.append(f"{s} FAIL {type(e).__name__}: {str(e)[:80]}")
        n_done += 1
        print(" | ".join(line), flush=True)
    if "lm" in models:
        try:
            models["lm"].close()
        except Exception:
            pass
    print(f"processed {n_done} sessions in {(time.time()-t_start)/60:.1f} min")
    for s in stages:
        print(f"cache {s}: {len([f for f in os.listdir(f'{FEAT}/{s}') if f.endswith('.json')])} ok")
    print("failures:", len(os.listdir(f"{FEAT}/fail")))


if __name__ == "__main__":
    main()

Writing outputs/claude/src/extract_session.py


In [31]:
apd.run("python outputs/claude/src/extract_session.py --limit 3 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO)'")

To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
malloc(): invalid size (unsorted)
Aborted (core dumped)

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/extract_session.py --limit 3 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO)'
[exit code 0, 109s, 3 lines; full log saved as outputs/claude/logs/cmd_0023.log]
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
malloc(): invalid size (unsorted)
Aborted (core dumped)
==================== END OF COPY ====================


0

In [32]:
apd.run("ls outputs/claude/cache/feat/*/ 2>/dev/null | head -20; for s in vision acoustic asr; do echo \"=== stage $s\"; python -X faulthandler outputs/claude/src/extract_session.py --limit 1 --stages $s 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO)' | tail -n 15; done")

outputs/claude/cache/feat/acoustic/:

outputs/claude/cache/feat/asr/:

outputs/claude/cache/feat/fail/:

outputs/claude/cache/feat/vision/:
s01359357.json
=== stage vision
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  1 s0258b2bc dur 174s | vision 60s [det 0.98 multi 0.00]
processed 1 sessions in 1.0 min
cache vision: 2 ok
failures: 0
=== stage acoustic
  1 s01359357 dur 567s | acoustic 9s [nan 0]
processed 1 sessions in 0.2 min
cache acoustic: 1 ok
failures: 0
=== stage asr

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3582.69it/s]
  1 s01359357 dur 567s | asr 121s [lang ru:0.98,ru:0.89,ru:0.93 wpm 92]
processed 1 sessions in 2.0 min
cache asr: 1 ok
failures: 0

==================== COPY FROM HERE INTO THE CHAT ====================
$ ls outputs/claude/cache/feat/*/ 2>/dev/null | head -20; for s in vision acoustic asr; do echo "=== stage $s"; python -X faulthandler outputs/claude/src/extract_ses

0

In [33]:
%%writefile outputs/claude/src/run_stages.sh
#!/bin/bash
# Runs each extraction stage in its own process (native libs corrupt the heap when co-loaded).
# usage: bash outputs/claude/src/run_stages.sh [extract_session.py args, e.g. --limit 5 or --sids a,b]
for s in vision acoustic asr; do
  echo "##### stage $s"
  python outputs/claude/src/extract_session.py --stages $s "$@" 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable|Loading weights)'
done

Writing outputs/claude/src/run_stages.sh


In [34]:
apd.run("printf '%s\\n' '' '## 2026-11-19 Phase 3 (build)' '- Single-process extraction of vision+acoustic+asr crashed with malloc(): invalid size (native heap corruption). Each stage alone works (vision 60 s, acoustic 9 s, asr 121 s per session; ~190 s total, ~11 h wall for 208 sessions). DECISION: run each stage in its own process (src/run_stages.sh). Stage order vision, acoustic, then asr; ASR window length/model to be reviewed against the 24 CPU-h cap (11 h wall x 2 cores = 22 CPU-h is too tight) once vision and acoustic are done.' >> outputs/claude/decisions.md; bash outputs/claude/src/run_stages.sh --sids sd49cc219,s2e121fb7")

##### stage vision
  1 s2e121fb7 dur 765s | vision 116s [det 1.00 multi 0.00]
  2 sd49cc219 dur 76s SHORT | vision 76s [det 1.00 multi 0.00]
processed 2 sessions in 3.2 min
cache vision: 4 ok
failures: 0
##### stage acoustic
  1 s2e121fb7 dur 765s | acoustic 7s [nan 0]
  2 sd49cc219 dur 76s SHORT | acoustic 5s [nan 0]
processed 2 sessions in 0.2 min
cache acoustic: 3 ok
failures: 0
##### stage asr

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1321.36it/s]
  1 s2e121fb7 dur 765s | asr 44s [lang en:0.35,en:0.35,en:0.35 wpm 0]
  2 sd49cc219 dur 76s SHORT | asr 90s [lang ru:0.99,ru:0.99,ru:0.99 wpm 138]
processed 2 sessions in 2.3 min
cache asr: 3 ok
failures: 0

==================== COPY FROM HERE INTO THE CHAT ====================
$ printf '%s\n' '' '## 2026-11-19 Phase 3 (build)' '- Single-process extraction of vision+acoustic+asr crashed with malloc(): invalid size (native heap corruption). Each stage alone works (vision 60 s, acoustic 9 s, asr 121 s per session; ~190 s tota

0

In [35]:
%%writefile outputs/claude/STATE.md
phase: 3 (build) in progress
done:
- Phases 1-2 complete (audit_notes.md, proposal.md, decisions.md, pinned requirements.txt)
- src/extract_session.py (stages vision/acoustic/asr, per-session cache in cache/feat/<stage>/, failures in cache/feat/fail/), src/run_stages.sh (one process per stage; co-loading native libs crashes)
- Tested on 5 sessions incl. two extreme ones (76 s; near-silent audio): all stages OK. Near-silent session has 0 ASR words -> treat acoustic+language as missing at assembly.
in progress: full extraction. Order: acoustic (all), vision (chunks of 40), then ASR after a budget decision. All steps resumable (cached per session).
next command: apd.run("python outputs/claude/src/extract_session.py --stages acoustic 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO)'") ; then repeat apd.run("python outputs/claude/src/extract_session.py --stages vision --limit 40 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)'") until it prints "processed 0 sessions"
open issues: ASR cost (~5-7 h wall) vs 24 CPU-h cap: decide model/window uniformly for ALL sessions after vision+acoustic; assembly must null acoustic/language for sessions with ~0 words or silent audio; luminance tertile is a stand-in; duration confound (~0.64)

Overwriting outputs/claude/STATE.md


In [36]:
apd.run("printf '%s\\n' '- Test run of extreme sessions: sd49cc219 (76 s, SHORT, overlapping windows) OK on all stages; s2e121fb7 (near-silent, -85 dB) vision OK but ASR 0 words; openSMILE returns values on silence, so acoustic/language for sessions with ~0 words are treated as MISSING (fold-wise median imputation), never as signal.' '- Budget rule: if ASR must be made cheaper (smaller Whisper model or fewer windows) the change applies to ALL 208 sessions, never to a subset, so train and test share one pipeline.' >> outputs/claude/decisions.md; tail -n 3 outputs/claude/decisions.md | cut -c1-120")

- Single-process extraction of vision+acoustic+asr crashed with malloc(): invalid size (native heap corruption). Each st
- Test run of extreme sessions: sd49cc219 (76 s, SHORT, overlapping windows) OK on all stages; s2e121fb7 (near-silent, -
- Budget rule: if ASR must be made cheaper (smaller Whisper model or fewer windows) the change applies to ALL 208 sessio

==================== COPY FROM HERE INTO THE CHAT ====================
$ printf '%s\n' '- Test run of extreme sessions: sd49cc219 (76 s, SHORT, overlapping windows) OK on all stages; s2e121fb7 (near-silent, -85 dB) vision OK but ASR 0 words; openSMILE returns values on silence, so acoustic/language for sessions with ~0 words are treated as MISSING (fold-wise median imputation), never as signal.' '- Budget rule: if ASR must be made cheaper (smaller Whisper model or fewer windows) the change applies to ALL 208 sessions, never to a subset, so train and test share one pipeline.' >> outputs/claude/decisions.md; tail -n 3 outputs/clau

0

In [37]:
apd.run("python outputs/claude/src/extract_session.py --stages acoustic 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO)' | tail -n 12")

197 sf69a7e79 dur 657s | acoustic 5s [nan 0]
198 sf761c6a4 dur 588s | acoustic 4s [nan 0]
199 sf799b3e3 dur 492s | acoustic 4s [nan 0]
200 sf8d0b0a6 dur 654s | acoustic 6s [nan 0]
201 sf9dd025e dur 804s | acoustic 5s [nan 0]
202 sfb536d56 dur 274s | acoustic 5s [nan 0]
203 sfce94f02 dur 637s | acoustic 6s [nan 0]
204 sfd60e573 dur 442s | acoustic 4s [nan 0]
205 sffc25315 dur 560s | acoustic 4s [nan 0]
processed 205 sessions in 36.9 min
cache acoustic: 208 ok
failures: 0

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/extract_session.py --stages acoustic 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO)' | tail -n 12
[exit code 0, 2214s, 12 lines; full log saved as outputs/claude/logs/cmd_0027.log]
197 sf69a7e79 dur 657s | acoustic 5s [nan 0]
198 sf761c6a4 dur 588s | acoustic 4s [nan 0]
199 sf799b3e3 dur 492s | acoustic 4s [nan 0]
200 sf8d0b0a6 dur 654s | acoustic 6s [nan 0]
201 sf9dd025e dur 804s | acoustic 5s [nan 0]
202 sfb536d56 dur 2

0

In [38]:
apd.run("python outputs/claude/src/extract_session.py --stages vision --limit 40 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)' | tail -n 15")

 29 s1d5f1357 dur 181s | vision 44s [det 1.00 multi 0.00]
 30 s1ed92662 dur 627s | vision 58s [det 1.00 multi 0.00]
 31 s2013399f dur 606s | vision 56s [det 1.00 multi 0.00]
 32 s2341bed4 dur 576s | vision 48s [det 0.98 multi 0.00]
 33 s239865ea dur 636s | vision 61s [det 1.00 multi 0.00]
 34 s2585a9c1 dur 534s | vision 52s [det 1.00 multi 0.00]
 35 s259173ab dur 518s | vision 46s [det 1.00 multi 0.00]
 36 s25b4d6cf dur 657s | vision 48s [det 1.00 multi 0.00]
 37 s25e2e230 dur 486s | vision 46s [det 1.00 multi 0.00]
 38 s26215f55 dur 528s | vision 48s [det 1.00 multi 0.00]
 39 s27d2a1b8 dur 473s | vision 47s [det 1.00 multi 0.00]
 40 s27d7c52e dur 396s | vision 44s [det 1.00 multi 0.00]
processed 40 sessions in 32.7 min
cache vision: 44 ok
failures: 0

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/extract_session.py --stages vision --limit 40 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)' | tail -n 15
[exit code 0, 1968s, 

0

In [41]:
apd.run("ls -la outputs/claude/src/; head -n 40 outputs/claude/src/train_cv.py | cut -c1-160; ls outputs/claude/cache/feat/vision | wc -l")

total 49
drwx------ 3 root root  4096 Nov 19 09:11 .
drwx------ 6 root root  4096 Nov 19 08:00 ..
drwx------ 3 root root  4096 Nov 19 07:25 audit
-rw------- 1 root root  2573 Nov 19 07:27 env_probe.py
-rw------- 1 root root 13385 Nov 19 07:45 extract_session.py
-rw------- 1 root root  2686 Nov 19 07:35 fetch_whisper.py
-rw------- 1 root root  3102 Nov 19 07:39 phase2_close.py
-rw------- 1 root root   410 Nov 19 07:52 run_stages.sh
-rw------- 1 root root  3544 Nov 19 07:31 tool_probe.py
-rw------- 1 root root  4811 Nov 19 09:11 train_cv.py
-rw------- 1 root root  2597 Nov 19 07:41 verify_tools.py
-rw------- 1 root root  1400 Nov 19 07:33 whisper_probe.py
"""Channel CV (participant-grouped, 5 folds x 5 seeds). Channels: acoustic (eGeMAPS), vision (blendshape/pose stats),
language (stats + e5 PCA). Only channels with complete cache are evaluated. Duration is NEVER a model input; it is used
only for confound diagnostics. Prints compact summaries; writes cache/cv_channels.csv."""
import os,

0

In [42]:
apd.run("python outputs/claude/src/extract_session.py --stages vision --limit 40 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)' | tail -n 15")

 29 s57268aa3 dur 504s | vision 332s [det 1.00 multi 0.00]
 30 s57d7058b dur 605s | vision 54s [det 1.00 multi 0.00]
 31 s5f73d41b dur 494s | vision 46s [det 1.00 multi 0.00]
 32 s6078f037 dur 600s | vision 60s [det 1.00 multi 0.00]
 33 s60827e8a dur 594s | vision 61s [det 1.00 multi 0.00]
 34 s62131203 dur 479s | vision 53s [det 1.00 multi 0.00]
 35 s62bba33d dur 944s | vision 62s [det 1.00 multi 0.00]
 36 s63c188e7 dur 422s | vision 55s [det 1.00 multi 0.00]
 37 s6594b089 dur 448s | vision 49s [det 1.00 multi 0.00]
 38 s66d4f7e4 dur 531s | vision 47s [det 1.00 multi 0.00]
 39 s678e8473 dur 453s | vision 54s [det 1.00 multi 0.00]
 40 s69f88191 dur 553s | vision 51s [det 1.00 multi 0.00]
processed 40 sessions in 38.8 min
cache vision: 84 ok
failures: 0

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/extract_session.py --stages vision --limit 40 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)' | tail -n 15
[exit code 0, 2331s,

0

In [46]:
apd.run("python outputs/claude/src/extract_session.py --stages vision --limit 45 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)' | tail -n 8")

  4 sfb536d56 dur 274s | vision 48s [det 0.99 multi 0.00]
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x2d0d9180] Detected moov in a free or hoov atom.
  5 sfce94f02 dur 637s | vision 43s [det 1.00 multi 0.00]
  6 sfd60e573 dur 442s | vision 45s [det 1.00 multi 0.00]
  7 sffc25315 dur 560s | vision 49s [det 1.00 multi 0.00]
processed 7 sessions in 6.8 min
cache vision: 208 ok
failures: 0

==================== COPY FROM HERE INTO THE CHAT ====================
$ python outputs/claude/src/extract_session.py --stages vision --limit 45 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable)' | tail -n 8
[exit code 0, 414s, 8 lines; full log saved as outputs/claude/logs/cmd_0035.log]
  4 sfb536d56 dur 274s | vision 48s [det 0.99 multi 0.00]
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x2d0d9180] Detected moov in a free or hoov atom.
  5 sfce94f02 dur 637s | vision 43s [det 1.00 multi 0.00]
  6 sfd60e573 dur 442s | vision 45s [det 1.00 multi 0.00]
  7 sffc25315 dur 560s | vision 49s [det 1.00 multi 0.00]
processed 7 sessions

0

In [47]:
apd.run("sed -n 41,140p outputs/claude/src/train_cv.py | cut -c1-170; echo '=== RUN'; python outputs/claude/src/train_cv.py 2>&1 | tail -n 30")


def channel_X(name):
    if name == "acoustic":
        d = load("acoustic")
    elif name == "vision":
        d = load("vision"); d = d[[c for c in d.columns if not c.startswith("qc_")]]
    else:
        d = load("asr"); d = d[[c for c in d.columns if not c.startswith("qc_")]]
    return d


def make_model(C, name, emb=None):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=C, max_iter=2000))


def oof_scores(X, y, groups, C, seed):
    skf = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    s = np.zeros(len(y))
    for a, b in skf.split(X, y, groups):
        # drop all-NaN / constant columns using train fold only
        keep = X[a].std(0) > 0 if not np.isnan(X[a]).any() else (~np.isnan(X[a]).all(0))
        m = make_model(C, "")
        m.fit(X[a][:, keep], y[a])
        s[b] = m.decision_function(X[b][:, keep])
    return s


def evaluate(name, X, ysub, dsub, Cs=(0.001, 0.003, 0.01, 0.03, 0.1)):
    y, g, ld = ysub.y.v

0

In [ ]:
apd.run("bash -c \"python outputs/claude/src/extract_session.py --stages asr --limit 30 2>&1 | grep -v -E '^(I0000|W0000|WARNING|INFO|To enable|Loading weights)' | tail -n 12\"")

first run

In [10]:
%%writefile outputs/claude/requirements.txt
numpy
pandas
scipy
scikit-learn
librosa
soundfile
faster-whisper

Overwriting outputs/claude/requirements.txt


In [11]:
apd.run("pip install -q -r outputs/claude/requirements.txt && python -c \"import numpy,pandas,scipy,sklearn,librosa,soundfile,faster_whisper,cv2;print('numpy',numpy.__version__,'pandas',pandas.__version__,'scipy',scipy.__version__,'sklearn',sklearn.__version__,'librosa',librosa.__version__,'soundfile',soundfile.__version__,'faster_whisper',faster_whisper.__version__,'cv2',cv2.__version__)\"")

numpy 2.4.6 pandas 2.2.3 scipy 1.16.3 sklearn 1.6.1 librosa 0.11.0 soundfile 0.14.0 faster_whisper 1.2.1 cv2 5.0.0

==================== COPY FROM HERE INTO THE CHAT ====================
$ pip install -q -r outputs/claude/requirements.txt && python -c "import numpy,pandas,scipy,sklearn,librosa,soundfile,faster_whisper,cv2;print('numpy',numpy.__version__,'pandas',pandas.__version__,'scipy',scipy.__version__,'sklearn',sklearn.__version__,'librosa',librosa.__version__,'soundfile',soundfile.__version__,'faster_whisper',faster_whisper.__version__,'cv2',cv2.__version__)"
[exit code 0, 15s, 1 lines; full log saved as outputs/claude/logs/cmd_0001.log]
numpy 2.4.6 pandas 2.2.3 scipy 1.16.3 sklearn 1.6.1 librosa 0.11.0 soundfile 0.14.0 faster_whisper 1.2.1 cv2 5.0.0
==================== END OF COPY ====================


0

In [12]:
# CELL from the model
%%writefile outputs/claude/src/audit/common.py
import os
import numpy as np
import pandas as pd

VID_EXT = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".mpg", ".mpeg", ".wmv", ".flv")
AUDIT_DIR = "outputs/claude/cache/audit"


def norm_id(x):
    return os.path.splitext(os.path.basename(str(x)))[0]


def list_videos(root):
    out = []
    for d, _, fs in os.walk(root):
        for f in sorted(fs):
            if f.lower().endswith(VID_EXT):
                out.append(os.path.join(d, f))
    return sorted(out)


def load_all():
    """Returns (labels, audit, files). All keyed by 'sid' = filename stem."""
    lab = pd.read_csv("data/train/train_labels.csv")
    lab["sid"] = lab["session_id"].map(norm_id)
    lab["cohort"] = lab["cohort"].astype(str).str.strip().str.capitalize()
    aud = pd.read_csv("data/audit.csv")
    key = next((c for c in ["session_id", "video", "filename", "file", "path", "video_path"]
                if c in aud.columns), aud.columns[0])
    aud["sid"] = aud[key].map(norm_id)
    files = list_videos("data/train") + list_videos("data/test")
    ft = pd.DataFrame({"path": files})
    ft["sid"] = ft["path"].map(norm_id)
    ft["split"] = np.where(ft["path"].str.startswith("data/train"), "train", "test")
    ft["ext"] = ft["path"].map(lambda p: os.path.splitext(p)[1].lower())
    ft["folder_cohort"] = ft["path"].map(
        lambda p: "Experimental" if "/Experimental/" in p else ("Control" if "/Control/" in p else ""))
    return lab, aud, ft

Writing outputs/claude/src/audit/common.py


In [13]:
# CELL from the model
%%writefile outputs/claude/src/audit/inventory.py
"""Phase 1 items 1, 2, 5: inventory, recording conditions, confound floor, strata."""
import os, sys, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from common import load_all, AUDIT_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
SEED = 0
os.makedirs(AUDIT_DIR, exist_ok=True)


def q(s):
    return s.dropna().astype(float).quantile([0, .25, .5, .75, 1]).round(2).tolist()


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


lab, aud, ft = load_all()

print("=== 1. INVENTORY ===")
print("audit.csv columns:", list(aud.columns))
print("files by split/ext:\n", ft.groupby(["split", "ext"]).size().to_string())
print("label rows:", len(lab), "| unique sid in labels:", lab.sid.nunique(),
      "| duplicate stems among all files:", int(ft.sid.duplicated().sum()))
print("labels without file:", int((~lab.sid.isin(ft.sid)).sum()),
      "| train files without label:", int((~ft[ft.split == "train"].sid.isin(lab.sid)).sum()),
      "| audit rows without file:", int((~aud.sid.isin(ft.sid)).sum()),
      "| files without audit row:", int((~ft.sid.isin(aud.sid)).sum()))
chk = ft[ft.split == "train"].merge(lab[["sid", "cohort"]], on="sid", how="inner")
print("folder-vs-csv cohort disagreements:", int((chk.folder_cohort != chk.cohort).sum()))

print("\nTrain sessions per cohort:\n", lab.cohort.value_counts().to_string())
print("Test sessions (files):", int((ft.split == "test").sum()))
pc = lab.groupby("cohort").participant_id.nunique()
print("Participants per cohort:\n", pc.to_string())
both = lab.groupby("participant_id").cohort.nunique()
print("Participants appearing in both cohorts:", int((both > 1).sum()), "of", lab.participant_id.nunique())
spp = lab.groupby("participant_id").size()
print("Sessions per participant (count of participants by n sessions):\n", spp.value_counts().sort_index().to_string())
print("Sessions per participant by cohort (mean, max):\n",
      lab.groupby(["cohort", "participant_id"]).size().groupby("cohort").agg(["mean", "max"]).round(2).to_string())

# merge audit with split / labels
df = aud.merge(ft[["sid", "split"]].drop_duplicates("sid"), on="sid", how="left")
df = df.merge(lab[["sid", "cohort", "participant_id"]], on="sid", how="left")
df.to_csv(f"{AUDIT_DIR}/merged.csv", index=False)
tr = df[df.split == "train"].dropna(subset=["cohort"]).copy()
te = df[df.split == "test"].copy()
tr["y"] = (tr.cohort == "Experimental").astype(int)
y = tr.y.values
g = tr.participant_id.values
print(f"\nMerged: train rows {len(tr)}, test rows {len(te)}")

for c in ["width", "height"]:
    pass
if {"width", "height"} <= set(df.columns):
    df["res"] = df.width.astype(int).astype(str) + "x" + df.height.astype(int).astype(str)
    print("Resolution (top 8) train:\n", df[df.split == "train"].res.value_counts().head(8).to_string())
    print("Resolution (top 8) test:\n", df[df.split == "test"].res.value_counts().head(8).to_string())
if "fps" in df.columns:
    print("fps rounded (train / test):\n",
          pd.concat([df[df.split == "train"].fps.round(0).value_counts().rename("train"),
                     df[df.split == "test"].fps.round(0).value_counts().rename("test")], axis=1).fillna(0).astype(int).head(8).to_string())
print("duration_s quantiles [min,q1,med,q3,max]  train:", q(tr.duration_s), " test:", q(te.duration_s))
for c in ["Experimental", "Control"]:
    print("  train", c, q(tr[tr.cohort == c].duration_s))

print("\n=== 2. RECORDING CONDITIONS vs COHORT (train) ===")
COND = [c for c in ["duration_s", "mean_luminance", "luminance_std", "face_detect_rate",
                    "audio_snr_db", "speech_fraction", "audio_rms_db"] if c in tr.columns]
rows = []
for c in COND:
    x = tr[c].astype(float)
    m = x.notna().values
    a, b = x[m & (y == 1)], x[m & (y == 0)]
    sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    d = (a.mean() - b.mean()) / sd if sd > 0 else np.nan
    p_s = stats.mannwhitneyu(a, b).pvalue
    auc = roc_auc_score(y[m], x[m])
    pm = tr.groupby("participant_id").agg(v=(c, "mean"), yy=("y", "max")).dropna()
    p_p = stats.mannwhitneyu(pm.v[pm.yy == 1], pm.v[pm.yy == 0]).pvalue
    rows.append(dict(var=c, n_missing=int((~m).sum()), q_Exp=q(a), q_Ctl=q(b), cohen_d=d, auroc_uni=auc,
                     p_session=p_s, p_participant=p_p))
res = pd.DataFrame(rows)
res["p_part_holm"] = holm(res.p_participant.values)
print("Quantile lists are [min,q1,med,q3,max]. p_session ignores clustering (anti-conservative); p_participant uses per-participant means.")
print(res[["var", "n_missing", "cohen_d", "auroc_uni", "p_session", "p_participant", "p_part_holm"]].round(4).to_string(index=False))
for _, r in res.iterrows():
    print(f"  {r['var']}: Exp {r.q_Exp} | Ctl {r.q_Ctl}")
res.to_csv(f"{AUDIT_DIR}/cond_vs_cohort.csv", index=False)

print("\nTrain vs test distribution shift (median train / median test / MWU p):")
for c in COND:
    a, b = tr[c].dropna(), te[c].dropna()
    if len(b):
        print(f"  {c}: {a.median():.3f} / {b.median():.3f} / p={stats.mannwhitneyu(a, b).pvalue:.3g}")

print("\n--- Confound floor: participant-grouped CV on audit.csv columns only ---")
tr["log_duration"] = np.log1p(tr.duration_s)
ALL = [c for c in ["log_duration", "width", "height", "fps", "mean_luminance", "luminance_std",
                   "face_detect_rate", "audio_snr_db", "speech_fraction", "audio_rms_db"] if c in tr.columns]
SETS = {"all_audit": ALL, "duration_only": ["log_duration"],
        "luminance_only": [c for c in ["mean_luminance", "luminance_std"] if c in tr.columns],
        "audio_only": [c for c in ["audio_snr_db", "audio_rms_db", "speech_fraction"] if c in tr.columns],
        "face_rate_only": [c for c in ["face_detect_rate"] if c in tr.columns],
        "format_only": [c for c in ["width", "height", "fps"] if c in tr.columns]}
n_part_min = int(pc.min())
NS = max(2, min(5, n_part_min))
print("folds:", NS, "| repeated over 5 seeds")


def lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))


def gb():
    return make_pipeline(SimpleImputer(strategy="median"),
                         GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, random_state=SEED))


def cv_auc(X, yy, gg, mk, seed):
    oof = np.zeros(len(yy))
    for a, b in StratifiedGroupKFold(n_splits=NS, shuffle=True, random_state=seed).split(X, yy, gg):
        m = mk()
        m.fit(X[a], yy[a])
        oof[b] = m.predict_proba(X[b])[:, 1]
    return roc_auc_score(yy, oof)


out = []
for name, cols in SETS.items():
    if not cols:
        continue
    X = tr[cols].astype(float).values
    for mname, mk in [("logreg", lr), ("gbm_depth2", gb)]:
        a = [cv_auc(X, y, g, mk, s) for s in range(5)]
        out.append(dict(features=name, model=mname, auroc_mean=np.mean(a), auroc_sd=np.std(a), n_features=len(cols)))
fl = pd.DataFrame(out)
print(fl.round(3).to_string(index=False))
fl.to_csv(f"{AUDIT_DIR}/confound_floor.csv", index=False)

# permutation null at participant level (only meaningful if participants are single-cohort)
rng = np.random.RandomState(SEED)
pl = tr.groupby("participant_id").y.max()
X = tr[ALL].astype(float).values
obs = np.mean([cv_auc(X, y, g, lr, s) for s in range(2)])
null = []
for _ in range(60):
    perm = pd.Series(rng.permutation(pl.values), index=pl.index)
    yp = tr.participant_id.map(perm).values
    null.append(cv_auc(X, yp, g, lr, 0))
null = np.array(null)
print(f"Permutation check (all_audit logreg, 60 participant-level label shuffles): observed {obs:.3f}, "
      f"null mean {null.mean():.3f} sd {null.std():.3f}, p={(1 + (null >= obs).sum()) / 61:.3f}")

print("\n=== 5. STRATA (audit.csv tertiles) ===")
for tcol in ["duration_tertile", "luminance_tertile", "snr_tertile"]:
    if tcol not in df.columns:
        print(tcol, "MISSING")
        continue
    t1 = pd.crosstab(tr[tcol], tr.cohort)
    t2 = te[tcol].value_counts().rename("test")
    tab = t1.join(t2, how="outer").fillna(0).astype(int)
    try:
        p = stats.chi2_contingency(t1.values)[1]
    except Exception:
        p = np.nan
    print(f"\n{tcol} (cohort x tertile chi2 p={p:.3g}; session-level):")
    print(tab.to_string())
    print("  value ranges of the underlying variable per tertile (train+test, min..max):")
    base = {"duration_tertile": "duration_s", "luminance_tertile": "mean_luminance", "snr_tertile": "audio_snr_db"}[tcol]
    print(df.groupby(tcol)[base].agg(["min", "max"]).round(2).to_string())
print("\nINVENTORY DONE; tables in", AUDIT_DIR)

Writing outputs/claude/src/audit/inventory.py


In [14]:
# CELL from the model
apd.run("python outputs/claude/src/audit/inventory.py")

=== 1. INVENTORY ===
audit.csv columns: ['session_id', 'path', 'duration_s', 'width', 'height', 'fps', 'audio_snr_db', 'speech_fraction', 'audio_rms_db', 'error', 'duration_tertile', 'snr_tertile', 'sid']
files by split/ext:
 split  ext 
test   .mp4     62
train  .mov      1
       .mp4    145
label rows: 146 | unique sid in labels: 146 | duplicate stems among all files: 0
labels without file: 0 | train files without label: 0 | audit rows without file: 0 | files without audit row: 0
folder-vs-csv cohort disagreements: 80

Train sessions per cohort:
 cohort
Elevated    80
Control     66
Test sessions (files): 62
Participants per cohort:
 cohort
Control     66
Elevated    80
Participants appearing in both cohorts: 0 of 146
Sessions per participant (count of participants by n sessions):
 1    146
Sessions per participant by cohort (mean, max):
           mean  max
cohort             
Control    1.0    1
Elevated   1.0    1

Merged: train rows 146, test rows 62
Resolution (top 8) train:
 r

1

Use Opus 5.5 with Claude Pro- limit is reached

In [17]:
apd.run("python outputs/claude/src/audit/inventory.py")

=== 1. INVENTORY ===
audit.csv columns: ['session_id', 'path', 'duration_s', 'width', 'height', 'fps', 'audio_snr_db', 'speech_fraction', 'audio_rms_db', 'error', 'duration_tertile', 'snr_tertile', 'sid']
files by split/ext:
 split  ext 
test   .mp4     62
train  .mov      1
       .mp4    145
label rows: 146 | unique sid in labels: 146 | duplicate stems among all files: 0
labels without file: 0 | train files without label: 0 | audit rows without file: 0 | files without audit row: 0
folder-vs-csv cohort disagreements: 0

Train sessions per cohort:
 cohort
Experimental    80
Control         66
Test sessions (files): 62
Participants per cohort:
 cohort
Control         66
Experimental    80
Participants appearing in both cohorts: 0 of 146
Sessions per participant (count of participants by n sessions):
 1    146
Sessions per participant by cohort (mean, max):
               mean  max
cohort                 
Control        1.0    1
Experimental   1.0    1

Merged: train rows 146, test rows 

0

In [18]:
apd.run("python outputs/claude/src/audit/luminance.py")

videos 208 | cached 0 | to do 208
  20/208
  40/208
  60/208
  80/208
  100/208
  120/208
  140/208
  160/208
  180/208
  200/208
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] Detected moov in a free or hoov atom.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 1, offset 0x26cb9d07: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 1, offset 0x26cb9d07: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 1, offset 0x26cb9d07: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 1, offset 0x26b9650a: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 1, offset 0x26a7574d: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 1, offset 0x2694dd06: partial file
[h264 @ 0x4c48740] Invalid NAL unit size (35583 > 27470).
[h264 @ 0x4c48740] missing picture in access unit with size 27474
[h264 @ 0x4b4bc80] Invalid NAL unit size (35583 > 27470).
[h264 @ 0x4b4bc80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x4b46880] stream 0, offset 0x2678cfb1: part

0

## Before hand-off

In [ ]:
#@title Check outputs
apd.check_contract()
apd.ls()